# OEP001041 — 全血 (peripheral blood leukocyte) RNA-seq 加齢トランスクリプトーム解析
Section 3.4 拡張: 連続年齢コホート(40–70歳)における `Expression ~ Age + Sex` の限界発現解析 + preranked GSEA

**実行環境**: Google Colab / Python runtime (Runtime > Change runtime type > Python 3)
Google Drive上の `metadata.txt` と `count.txt` を読み込み、R (`rpy2`経由) で edgeR TMM → limma-voom → fgsea(msigdbr) を実行します。

> GSE262619側の解析と同じ MSigDB / msigdbr バージョンを使うことを忘れずに(ノートブック末尾でバージョンを出力します)。


> **v11 変更点**: 査読対応として、血球組成交絡(cell-type composition confound)への
> 感度分析を2箇所に追加した。
> 1. §10.5: 全体のAge差次発現モデルに好中球/リンパ球マーカー由来の転写プロキシ
>    (NLR_proxy)を共変量として追加し、Age効果t統計量が補正前後でどの程度一致するかを確認する。
> 2. §16.6: §16で事前specifiedした代表4パスウェイ(IFN/OXPHOS/Proteostasis x2)について、
>    NLR_proxy補正後のdavies検定p値・ΔAIC・transition age点推定を、未補正の結果(§16.2-16.3)
>    と並べて比較する。
>
> また、タイトルおよびData Availability記載の "PBMC" は、原論文(Gao, Li, & Cai, 2025,
> *J Gerontol A*, glaf054)のタイトルが "Human **Peripheral Blood Leukocyte** Transcriptome-Based
> Aging Clock" であることに基づき、"whole-blood (peripheral blood leukocyte)" に修正した
> (PBMCではなく好中球を含む全血であるため、NLR型の交絡がより直接的に関係する)。

## 1. Google Drive のマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. データの場所を指定
`DRIVE_DIR` を、`metadata.txt` と `count.txt` を置いたフォルダのパスに書き換えてください。
例: `/content/drive/MyDrive/OEP001041`

In [ ]:
import os

def find_file(root, filename, max_hits=10):
    """root配下を再帰的に探索してfilenameに一致するファイルのフルパスを返す"""
    hits = []
    for dirpath, dirnames, filenames in os.walk(root):
        if filename in filenames:
            hits.append(os.path.join(dirpath, filename))
            if len(hits) >= max_hits:
                break
    return hits

# まず候補パスを試す(ここは書き換えてもOK)
DRIVE_DIR = "/content/drive/MyDrive/OEP001041"
meta_path = os.path.join(DRIVE_DIR, "metadata.txt")
count_path = os.path.join(DRIVE_DIR, "count.txt")

if not (os.path.exists(meta_path) and os.path.exists(count_path)):
    print(f"指定パス {DRIVE_DIR} には見つからないため、Drive全体を検索します(数十秒〜数分かかる場合があります)...")
    meta_hits = find_file("/content/drive/MyDrive", "metadata.txt")
    count_hits = find_file("/content/drive/MyDrive", "count.txt")
    print("metadata.txt candidates:", meta_hits)
    print("count.txt candidates:", count_hits)

    if meta_hits and count_hits:
        meta_path = meta_hits[0]
        count_path = count_hits[0]
        print("\n見つかったファイルを使用します。")
        if len(meta_hits) > 1 or len(count_hits) > 1:
            print("※ 複数候補が見つかりました。誤っている場合は meta_path / count_path を手動で上書きしてください。")
    else:
        raise FileNotFoundError(
            "metadata.txt / count.txt がGoogle Drive内に見つかりませんでした。\n"
            "Colab左パネルの「ファイル」アイコン → drive/MyDrive を開いてアップロード先を確認し、\n"
            "meta_path・count_path を直接指定してください(例: meta_path='/content/drive/MyDrive/フォルダ名/metadata.txt')。"
        )

print("\nmeta_path :", meta_path)
print("count_path:", count_path)


## 3. rpy2 のセットアップ(R連携)
ColabにはRとrpy2が入っていない場合があるため、必要ならインストールします。

In [ ]:
# R本体の確認・インストール(数分かかる場合があります)
import subprocess
r_check = subprocess.run(["which", "R"], capture_output=True, text=True).stdout.strip()
if not r_check:
    print("Rをインストールしています...")
    !apt-get install -y r-base-core -qq > /dev/null
else:
    print("R found at:", r_check)

!pip install -q rpy2
%load_ext rpy2.ipython


## 4. R パッケージのインストール
`edgeR`, `limma`, `fgsea`, `msigdbr`, `dplyr` をBioconductor/CRANから導入します。初回のみ数分〜十数分かかります。

In [ ]:
%%R
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager", repos = "https://cloud.r-project.org")
BiocManager::install(c("edgeR", "limma", "fgsea"), update = FALSE, ask = FALSE)
install.packages(c("msigdbr", "dplyr"), repos = "https://cloud.r-project.org")

suppressPackageStartupMessages({
  library(edgeR)
  library(limma)
  library(fgsea)
  library(msigdbr)
  library(dplyr)
})
cat("packages loaded\n")


## 5. Pythonの変数(ファイルパス)をRへ受け渡す

In [ ]:
%%R -i meta_path -i count_path
cat("metadata:", meta_path, "\n")
cat("counts  :", count_path, "\n")


## 6. データ読み込み・整列
- `metadata.txt`: `-` と `NA` はどちらも欠損として扱います
- `count.txt`: 遺伝子(ENSG ID) × サンプルID の生カウント行列
- サンプルIDの完全一致を確認したうえで、列順をmetadataに合わせます

In [ ]:
%%R
meta <- read.delim(meta_path, na.strings = c("NA", "-"), check.names = FALSE, stringsAsFactors = FALSE)
counts_all <- read.delim(count_path, row.names = 1, check.names = FALSE)

meta$sample <- as.character(meta$sample)
colnames(counts_all) <- as.character(colnames(counts_all))

cat("metadata N =", nrow(meta), " / count.txt N =", ncol(counts_all), "\n")
common <- intersect(meta$sample, colnames(counts_all))
cat("共通サンプル数 =", length(common), "\n")
stopifnot(length(common) == nrow(meta), length(common) == ncol(counts_all))

meta <- meta[match(colnames(counts_all), meta$sample), ]
stopifnot(all(meta$sample == colnames(counts_all)))
cat("サンプル順の整列: OK\n")


## 7. 40–70歳サブセットの抽出
HRV由来のピーク年齢(CHI=44.9y, α1=57.3y, α2=67.1y)をすべて含む区間です。
Age・Genderに欠損がある場合はここで除外されます。

In [ ]:
%%R
keep <- which(meta$Age >= 40 & meta$Age <= 70 & !is.na(meta$Age) & !is.na(meta$Gender))
meta_sub <- meta[keep, ]
counts_sub <- counts_all[, keep]
meta_sub$Gender <- factor(meta_sub$Gender, levels = c("F", "M"))

cat("N(40-70y, 欠損除外後) =", ncol(counts_sub), "\n")
print(table(meta_sub$Gender))
cat("Age range:", range(meta_sub$Age), "\n")

age_sorted <- sort(meta_sub$Age)
cat("最大年齢ギャップ:", max(diff(age_sorted)), "歳\n")


## 8. QC — ライブラリサイズ

In [ ]:
%%R
libsize <- colSums(counts_sub)
cat("Library size (M reads) summary:\n")
print(summary(libsize / 1e6))


## 9. edgeR: フィルタリング + TMM正規化

In [ ]:
%%R
dge <- DGEList(counts = counts_sub)
keep_genes <- filterByExpr(dge)
dge <- dge[keep_genes, , keep.lib.sizes = FALSE]
dge <- calcNormFactors(dge, method = "TMM")
cat("フィルタ後 遺伝子数:", nrow(dge), "\n")


## 10. limma-voom: `Expression ~ Age + Sex`

In [ ]:
%%R
design <- model.matrix(~ Age + Gender, data = meta_sub)
v <- voom(dge, design, plot = TRUE)
fit <- lmFit(v, design)
fit <- eBayes(fit)

res <- topTable(fit, coef = "Age", number = Inf, sort.by = "P")
res$gene_ensembl <- rownames(res)
res$gene <- sub("\\..*", "", res$gene_ensembl)

cat("\nFDR<0.05:", sum(res$adj.P.Val < 0.05), "\n")
cat("FDR<0.25:", sum(res$adj.P.Val < 0.25), "\n")
cat("nominal p<0.01:", sum(res$P.Value < 0.01), "\n")
head(res, 15)


## 10.5 血球組成サロゲート(NLR-proxy)による補正感度分析

[v11追加] 3.4節の議論で指摘した限界(血球組成交絡)への対応。GSE262619/OEP001041ともにCBC(complete blood count)や細胞タイプ分画推定値は提供されていないため、確立された好中球・リンパ球マーカー遺伝子パネルから加齢に伴う血球組成シフト(Neutrophil-to-Lymphocyte Ratio, NLRの転写プロキシ)を近似し、Age効果がこの組成シフトのみで説明されるものではないことを確認する。

> **方法**: 各サンプルについて、好中球マーカー遺伝子群(FCGR3B, CSF3R, CXCR2, S100A8, S100A9, MMP9)とリンパ球マーカー遺伝子群(CD3D, CD3E, CD2, CD19, MS4A1, KLRB1)の voom log2-CPM を遺伝子ごとにz-score化した上で平均し、`NLR_proxy = mean(好中球z) − mean(リンパ球z)` を計算する。このNLR_proxyを共変量に追加したモデル(`~ Age + Gender + NLR_proxy`)でlimma-voomを再実行し、Age係数(t統計量・FDR)が未補正モデルとどの程度一致するかを比較する。マーカー遺伝子はPeters et al. (2015, *Nat Commun* 6:8570)等、全血加齢トランスクリプトーム研究で用いられる古典的な好中球・リンパ球系統マーカーに基づく。CIBERSORTx等の分画推定ツールは参照シグネチャ行列や追加の重い依存パッケージを要し、Colab上でのビルド安定性の観点から16-A注記(GSVAを見送った理由)と同じ判断で見送り、透明性の高い軽量なマーカースコアで代替する。
>
> Ensembl gene IDへのマッピングはノートブック内にハードコードせず、実行時に`org.Hs.eg.db`で解決する(検証可能性の担保)。マーカーパネル自体の妥当性は、査読対応・SI記載の際に改めてBioMart等で再確認することを推奨する。

In [ ]:
%%R
# --- 10-5-A: org.Hs.eg.dbのインストールとマーカー遺伝子のEnsembl IDへのマッピング ---
if (!requireNamespace("org.Hs.eg.db", quietly = TRUE)) {
  BiocManager::install("org.Hs.eg.db", update = FALSE, ask = FALSE)
}
suppressPackageStartupMessages({
  library(org.Hs.eg.db)
  library(AnnotationDbi)
})

neutrophil_symbols <- c("FCGR3B", "CSF3R", "CXCR2", "S100A8", "S100A9", "MMP9")
lymphocyte_symbols <- c("CD3D", "CD3E", "CD2", "CD19", "MS4A1", "KLRB1")

map_symbol_to_ensembl <- function(symbols) {
  m <- suppressMessages(AnnotationDbi::select(org.Hs.eg.db, keys = symbols,
                                               keytype = "SYMBOL", columns = "ENSEMBL"))
  m <- m[!is.na(m$ENSEMBL) & !duplicated(m$SYMBOL), ]
  rownames(m) <- NULL
  m
}

neutrophil_map <- map_symbol_to_ensembl(neutrophil_symbols)
lymphocyte_map <- map_symbol_to_ensembl(lymphocyte_symbols)

cat("=== 好中球マーカー: symbol -> Ensembl ID ===\n"); print(neutrophil_map)
cat("\n=== リンパ球マーカー: symbol -> Ensembl ID ===\n"); print(lymphocyte_map)

# カウント行列(バージョン無しEnsembl ID)に実際に存在するかを確認
count_genes_bare <- unique(sub("\\..*", "", rownames(dge)))
neutrophil_map$in_matrix <- neutrophil_map$ENSEMBL %in% count_genes_bare
lymphocyte_map$in_matrix <- lymphocyte_map$ENSEMBL %in% count_genes_bare

cat("\n好中球マーカー: カウント行列中に", sum(neutrophil_map$in_matrix), "/", nrow(neutrophil_map), "件が存在\n")
cat("リンパ球マーカー: カウント行列中に", sum(lymphocyte_map$in_matrix), "/", nrow(lymphocyte_map), "件が存在\n")
if (any(!neutrophil_map$in_matrix)) print(neutrophil_map[!neutrophil_map$in_matrix, ])
if (any(!lymphocyte_map$in_matrix)) print(lymphocyte_map[!lymphocyte_map$in_matrix, ])

# 各セットにつき最低3遺伝子がマッチしていることを要求する(単一遺伝子への依存を避ける)
stopifnot(sum(neutrophil_map$in_matrix) >= 3, sum(lymphocyte_map$in_matrix) >= 3)
cat("\nOK: 両マーカーセットとも最低3遺伝子がマッチしました。\n")

In [ ]:
%%R
# --- 10-5-B: voom log2-CPMからNLR_proxyを計算 ---
expr10 <- v$E
rownames(expr10) <- sub("\\..*", "", rownames(expr10))
stopifnot(identical(colnames(expr10), meta_sub$sample))

# 遺伝子ごとにサンプル間z-score化(16-Cと同じ考え方)
gmean10 <- rowMeans(expr10)
gsd10 <- apply(expr10, 1, sd)
gsd10[gsd10 == 0] <- NA
expr10_z <- (expr10 - gmean10) / gsd10

neu_genes <- neutrophil_map$ENSEMBL[neutrophil_map$in_matrix]
lym_genes <- lymphocyte_map$ENSEMBL[lymphocyte_map$in_matrix]

neutrophil_score <- colMeans(expr10_z[neu_genes, , drop = FALSE], na.rm = TRUE)
lymphocyte_score <- colMeans(expr10_z[lym_genes, , drop = FALSE], na.rm = TRUE)
meta_sub$NLR_proxy <- neutrophil_score - lymphocyte_score

cat("NLR_proxy summary:\n"); print(summary(meta_sub$NLR_proxy))

age_cor <- cor.test(meta_sub$Age, meta_sub$NLR_proxy, method = "spearman")
cat(sprintf("\nNLR_proxy と Age の相関 (Spearman rho = %.3f, p = %.4g)\n",
            unname(age_cor$estimate), age_cor$p.value))
cat("(正の相関は、加齢に伴う好中球優位シフト[免疫老化の既知パターン]と整合的)\n")

In [ ]:
%%R
# --- 10-5-C: NLR_proxyを共変量に加えた補正モデルの再フィット ---
# out_dir はSection 14で定義されるが、このセルはそれより前に実行されるため、
# ここでも同じ定義を(冪等に)行っておく。
out_dir <- file.path(dirname(meta_path), "results")
dir.create(out_dir, showWarnings = FALSE)

design_adj <- model.matrix(~ Age + Gender + NLR_proxy, data = meta_sub)
v_adj <- voom(dge, design_adj, plot = TRUE)
fit_adj <- lmFit(v_adj, design_adj)
fit_adj <- eBayes(fit_adj)

res_adj <- topTable(fit_adj, coef = "Age", number = Inf, sort.by = "P")
res_adj$gene_ensembl <- rownames(res_adj)
res_adj$gene <- sub("\\..*", "", res_adj$gene_ensembl)

cat("=== NLR_proxy補正後: Age係数 ===\n")
cat("FDR<0.05:", sum(res_adj$adj.P.Val < 0.05), "(補正前:", sum(res$adj.P.Val < 0.05), ")\n")
cat("FDR<0.25:", sum(res_adj$adj.P.Val < 0.25), "(補正前:", sum(res$adj.P.Val < 0.25), ")\n")
cat("nominal p<0.01:", sum(res_adj$P.Value < 0.01), "(補正前:", sum(res$P.Value < 0.01), ")\n")

In [ ]:
%%R
# --- 10-5-D: 補正前後のAge効果 t統計量の一致度 ---
common_genes10 <- intersect(res$gene, res_adj$gene)
t_orig <- setNames(res$t, res$gene)[common_genes10]
t_adj  <- setNames(res_adj$t, res_adj$gene)[common_genes10]

t_cor <- cor(t_orig, t_adj, method = "pearson")
cat(sprintf("Age効果t統計量の相関(補正前 vs 補正後, n=%d genes): r = %.4f\n",
            length(common_genes10), t_cor))

top_n <- 200
top_orig <- names(sort(abs(t_orig), decreasing = TRUE))[1:top_n]
top_adj  <- names(sort(abs(t_adj),  decreasing = TRUE))[1:top_n]
jaccard_top <- length(intersect(top_orig, top_adj)) / length(union(top_orig, top_adj))
cat(sprintf("上位%d遺伝子(|t|降順)のJaccard一致度: %.3f\n", top_n, jaccard_top))

cat("\n解釈の目安: t統計量の相関が高く(例: r>0.9)、上位遺伝子の重なりが大きいほど、\n",
    "Age効果はNLR_proxy(血球組成サロゲート)で説明される部分が小さいことを示唆する。\n",
    "逆に相関が大きく低下する場合、3.4節の知見の少なくとも一部は組成交絡由来である\n",
    "可能性を積極的に検討する必要がある。\n", sep = "")

comparison_df10 <- data.frame(
  metric = c("N_FDR05", "N_FDR25", "N_nominal_p01", "t_stat_correlation", "top200_jaccard"),
  unadjusted = c(sum(res$adj.P.Val < 0.05), sum(res$adj.P.Val < 0.25),
                 sum(res$P.Value < 0.01), NA, NA),
  NLR_adjusted = c(sum(res_adj$adj.P.Val < 0.05), sum(res_adj$adj.P.Val < 0.25),
                    sum(res_adj$P.Value < 0.01), t_cor, jaccard_top)
)
print(comparison_df10)
write.csv(comparison_df10, file.path(out_dir, "OEP001041_NLRproxy_DE_sensitivity.csv"), row.names = FALSE)
cat("\n保存しました:", file.path(out_dir, "OEP001041_NLRproxy_DE_sensitivity.csv"), "\n")

## 11. GSEA用ランクリスト作成

In [ ]:
%%R
ranks <- setNames(res$t, res$gene)
ranks <- ranks[!duplicated(names(ranks))]
ranks <- sort(ranks, decreasing = TRUE)
cat("ranked genes:", length(ranks), "\n")


## 12. MSigDB 遺伝子セット取得(Hallmark / Reactome / KEGG_MEDICUS / GO:BP)
`msigdbr` がオンラインからダウンロードします。

> 修正メモ: `"CP:KEGG"` は現行のmsigdbrでは廃止され `CP:KEGG_LEGACY` / `CP:KEGG_MEDICUS` に分割されています。未対応のままだと `get_pathways("C2", "CP:KEGG")` の行で `Unknown subcollection.` エラーが出て以降(GO:BP)のセルまで到達しません。

In [ ]:
%%R
# 参考: 手元の環境で実際に有効なsubcollection名を確認したい場合は
# print(msigdbr_collections(), n = 30) を先に実行してください。

# msigdbrは初回呼び出し時にZenodoから遺伝子セットデータをダウンロードします。
# Zenodo側が一時的に混雑/タイムアウト(504等)することがあるため、簡単なリトライを挟みます。
# 一度ローカルキャッシュにダウンロードできれば、以降の呼び出しはネットワーク不要になります。
with_retry <- function(expr_fun, max_tries = 5, base_wait = 10) {
  for (attempt in seq_len(max_tries)) {
    result <- tryCatch(list(ok = TRUE, value = expr_fun()),
                        error = function(e) list(ok = FALSE, err = e))
    if (result$ok) return(result$value)
    if (attempt == max_tries) stop(result$err)
    wait_s <- base_wait * attempt
    cat(sprintf("  [試行 %d/%d 失敗: %s] %d秒待って再試行します...\n",
                attempt, max_tries, conditionMessage(result$err), wait_s))
    Sys.sleep(wait_s)
  }
}

get_pathways <- function(collection, subcollection = NULL) {
  m <- with_retry(function() {
    if (is.null(subcollection)) {
      msigdbr(species = "Homo sapiens", collection = collection)
    } else {
      msigdbr(species = "Homo sapiens", collection = collection, subcollection = subcollection)
    }
  })
  split(x = m$ensembl_gene, f = m$gs_name)
}

cat("Hallmark取得中...\n");  pathways_hallmark <- get_pathways("H")
cat("Reactome取得中...\n");  pathways_reactome <- get_pathways("C2", "CP:REACTOME")
# "CP:KEGG" は廃止済み。再現性重視ならCP:KEGG_LEGACY、最新の代謝経路網羅を取りたい場合はCP:KEGG_MEDICUSを使用。
# GSE262619側の解析と揃える場合は、そちらで使ったKEGGバージョンに合わせて下の行を変更してください。
cat("KEGG(medicus)取得中...\n"); pathways_kegg <- get_pathways("C2", "CP:KEGG_MEDICUS")
cat("GO:BP取得中...\n");   pathways_gobp <- get_pathways("C5", "GO:BP")

cat("Hallmark:", length(pathways_hallmark),
    " Reactome:", length(pathways_reactome),
    " KEGG(medicus):", length(pathways_kegg),
    " GO:BP:", length(pathways_gobp), "\n")

## 13. preranked GSEA(コレクションごとに独立実行)

In [ ]:
%%R
# [v7追加] fgsea(eps=0)はadaptive multilevel splittingを用いるため確率的であり、
# re-run毎にNES/pval/padjが微小に変動する(結論自体は不変でも、padj順で
# 上位N件を切り出す下流の候補パスウェイ選定は境界付近で入れ替わりうる)。
# 再現性確保のため、GSEA本体の呼び出し直前にseedを固定する。
set.seed(42)

run_fgsea <- function(pathways, ranks, label) {
  r <- fgsea(pathways = pathways, stats = ranks, minSize = 10, maxSize = 500, eps = 0)
  r$collection <- label
  r[order(r$pval), ]
}

res_hallmark <- run_fgsea(pathways_hallmark, ranks, "Hallmark")
res_reactome <- run_fgsea(pathways_reactome, ranks, "Reactome")
res_kegg     <- run_fgsea(pathways_kegg, ranks, "KEGG")
res_gobp     <- run_fgsea(pathways_gobp, ranks, "GO_BP")

cat("=== 有意パスウェイ数 (FDR<0.25) ===\n")
cat("Hallmark:", sum(res_hallmark$padj < 0.25, na.rm = TRUE), "\n")
cat("Reactome:", sum(res_reactome$padj < 0.25, na.rm = TRUE), "\n")
cat("KEGG:", sum(res_kegg$padj < 0.25, na.rm = TRUE), "\n")
cat("GO_BP:", sum(res_gobp$padj < 0.25, na.rm = TRUE), "\n")


## 14. 結果の保存(Google Driveへ書き戻し)

In [ ]:
%%R
out_dir <- file.path(dirname(meta_path), "results")
dir.create(out_dir, showWarnings = FALSE)

write.csv(res, file.path(out_dir, "OEP001041_DE_Age_40to70_limma.csv"), row.names = FALSE)

save_gsea <- function(df, name) {
  df2 <- df
  df2$leadingEdge <- vapply(df2$leadingEdge, function(x) paste(x, collapse = ";"), character(1))
  write.csv(df2, file.path(out_dir, paste0("OEP001041_GSEA_", name, "_40to70.csv")), row.names = FALSE)
}
save_gsea(res_hallmark, "Hallmark")
save_gsea(res_reactome, "Reactome")
save_gsea(res_kegg, "KEGG")
save_gsea(res_gobp, "GOBP")

cat("保存先:", out_dir, "\n")
cat("msigdbr version:", as.character(packageVersion("msigdbr")), "\n")
cat("fgsea version:", as.character(packageVersion("fgsea")), "\n")


## 15. トップパスウェイの確認(炎症・酸化ストレス・ミトコンドリア・老化関連テーマ)
GSE262619のTable 1と同じ4テーマで上位パスウェイを確認し、方向性(NESの符号)を比較してください。

In [ ]:
%%R
for (r in list(res_hallmark, res_reactome, res_kegg, res_gobp)) {
  print(head(r[, c("pathway","NES","pval","padj","collection")], 5))
}


## 16. パスウェイ単位の非線形年齢軌跡と transition age の独立推定

Section 10–13 の `Expression ~ Age + Gender` は年齢効果を**線形**と仮定した限界発現解析でした。しかし老化に伴うパスウェイ活性の変化は単調線形とは限らず、ある年齢帯で急激に変化する非線形(閾値的・breakpoint的)な軌跡を取ることが知られています(IFN応答の急上昇、OXPHOS/ミトコンドリア機能の緩やかな低下、proteostasis(UPR・シャペロン・プロテアソーム)の破綻など)。

以下では各パスウェイについて、特定のtransition ageを仮定せず、データから独立に(post-hocに)推定します:

1. Hallmark / Reactome / KEGG_MEDICUS / GO:BP から IFN・OXPHOS・proteostasis 関連の遺伝子セットをキーワードで抽出
2. サンプルごとのパスウェイ活性を、メンバー遺伝子のz-scoreの平均(mean gene z-score)で推定(voom正規化log-CPMを入力)
3. パスウェイごとに `score ~ Age + Gender` の breakpoint回帰(`segmented`パッケージ)を当てはめ、**transition age(変曲年齢)** を推定
   - 複数の初期値で頑健にフィットし、AICが最良のものを採用
   - davies検定で「線形からの逸脱(非線形性の存在)」自体を検定
   - ケースリサンプリングブートストラップでtransition ageの95%CIを算出
4. 独立した交差検証として GAM(`mgcv::gam`, thin-plate smooth)を当てはめ、平滑化曲線の傾きが最大になる年齢(steepest-slope age)をbreakpoint推定と付き合わせる

※ 横断データであるため、ここで得られる「transition age」はコホート内の年齢層間の差を反映したものであり、個人の縦断的軌跡そのものではない点に注意してください。

※ **手法メモ(スコアリング方法について)**: 当初はssGSEA(`GSVA`パッケージ)の使用を想定していましたが、現行のGSVA(バージョン2.x系)は`SpatialExperiment`・`magick`(ImageMagick)等の重量級依存パッケージを新たに要求するようになっており、Colab環境ではこれらのソースビルドに失敗することを確認しました(システムライブラリの追加導入が必要)。そこで、追加のシステム/Rパッケージ依存を発生させない標準的な代替として、**メンバー遺伝子のz-scoreの平均**によるサンプルごとのパスウェイ活性スコアを採用しています。年齢に対する非線形性・breakpointの検出という本セクションの目的においては、スコアリング手法の選択が結論を大きく左右する可能性は低いと考えられますが、厳密な感度分析としてssGSEAとの一致を確認したい場合は、Colab側でImageMagick開発ライブラリ(`apt-get install -y libmagick++-dev`等)を導入した上でGSVAを別途インストールし、本セクションの`pathway_scores`をssGSEA版に差し替えて再実行することで比較できます。

In [ ]:
%%R
# --- 16-A: 追加パッケージのインストール・ロード ---
# (GSVAは16.のイントロで説明の通り、Colabでの依存パッケージビルド失敗を避けるため使用しません)
needed_cran <- c("segmented", "mgcv", "ggplot2")
installed <- rownames(installed.packages())
to_install_cran <- setdiff(needed_cran, installed)

if (length(to_install_cran) > 0) {
  install.packages(to_install_cran, repos = "https://cloud.r-project.org")
}

suppressPackageStartupMessages({
  library(segmented)
  library(mgcv)
  library(ggplot2)
})
cat("segmented:", as.character(packageVersion("segmented")),
    " mgcv:", as.character(packageVersion("mgcv")),
    " ggplot2:", as.character(packageVersion("ggplot2")), "\n")

### 16.1 IFN・OXPHOS・Proteostasis 関連パスウェイの抽出

既にロード済みの `pathways_hallmark` / `pathways_reactome` / `pathways_kegg` / `pathways_gobp` から、キーワードに一致する遺伝子セットを収集します。GSEAで実際に検定された(`tested_pathways`)候補に限定したうえで、疾患variant派生パスウェイの除外、Jaccard類似度(閾値0.8)による重複統合を行います。**v8では、この後の「カテゴリごと上位N件で打ち切る」処理(旧`MAX_PER_CATEGORY`)を撤廃しました。** Jaccard整理後の全候補についてカテゴリ単位のFDRエビデンス表 `category_evidence_df`(PRIMARY_FDR=0.05・SENSITIVITY_FDR=0.25での有意数とNES符号の方向一致を集計、`OEP001041_CategoryLevelEvidence_FDR.csv` として保存)を作成し、証拠の全体像を記録します。そのうえで、16.2以降のper-pathway breakpoint/GAM解析には、事前specified(=fgseaの結果を見る前に決めた)代表パスウェイ4つ(IFN・OXPHOS各1、Proteostasis 2)のみを投入します(詳細は本セル末尾の`representative_spec`と、下記v8ノート参照)。

> **v6修正**: 以前のバージョンでは、候補パスウェイの検索母集団が `pathways_gobp` 等(msigdbrから取得した全遺伝子セット、fgseaのフィルタ前)になっており、`fgsea(minSize=10, maxSize=500)` を通過せずGSEAで検定されなかった小さい遺伝子セット(例: `GOBP_MITOCHONDRIAL_RESPIRASOME_ASSEMBLY`, n_genes=9)が非線形transition-age解析の対象に紛れ込みうる不整合があった。v6では候補選定を `tested_pathways`(NES/pval/padjが全て非NAの、実際にfgseaで検定済みの遺伝子セット)に限定し、`candidate pathway ⊆ GSEA-tested pathway` を保証する(セル末尾に `stopifnot` によるsanity checkを追加)。
>
> Methods記載案(v6): *All pathways entering downstream transition-age analyses were required to have passed the same gene-set size filters (minSize = 10, maxSize = 500) used for the corresponding preranked GSEA, i.e., candidate pathways were restricted to the set of gene sets with non-missing NES, nominal p-value, and BH-adjusted q-value in fgsea.*
>
> **v8修正 (1) — MAX_PER_CATEGORYによる打ち切りの撤廃**: v6/v7では、Jaccard整理後の候補をfgsea `padj` 昇順・遺伝子セットサイズ降順で並べ替え、各カテゴリ上位 `MAX_PER_CATEGORY`(=8)個のみをdownstream解析に投入していた。この打ち切りには (a) 「8個」という基準自体の恣意性、(b) 上位8件に入らなかった候補の黙示的な除外、という2つの問題があり、頑健性への疑義(基準を変えたら結果は変わるのか)を招きうる。v8ではこの打ち切りロジックを完全に削除し、Jaccard整理後の**全候補**を対象に、カテゴリごとの有意パスウェイ数(PRIMARY_FDR=0.05 / SENSITIVITY_FDR=0.25の2水準)と、有意パスウェイ間でのNES符号(エンリッチメント方向)の一致度を集計した `category_evidence_df` を作成し、`OEP001041_CategoryLevelEvidence_FDR.csv` として保存する。
>
> **v8修正 (2) — 代表パスウェイの事前specification(post-hoc選択ではない)**: 16.2以降のtransition-age解析(breakpoint回帰・GAM・ブートストラップ)は計算コストが高く、候補全件に適用すると多重検定・計算時間の両面で現実的でない。そこでv8では、per-pathway解析に投入する遺伝子セットを、**上記category_evidence_df(fgseaのpadjランキングや有意数の集計結果)を参照する前に**、各カテゴリの中核的な生物学的プロセスを代表する標準的なMSigDB遺伝子セットとして次の4つに固定した:
> - IFN: `HALLMARK_INTERFERON_ALPHA_RESPONSE`
> - OXPHOS: `HALLMARK_OXIDATIVE_PHOSPHORYLATION`
> - Proteostasis: `REACTOME_AUTOPHAGY` および `KEGG_MEDICUS_REFERENCE_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION`
>
> これらはfgseaのpadjが小さかったから選んだものではない(実際、v6/v7のMAX_PER_CATEGORY打ち切りロジックで選ばれていたパスウェイの一部はここでは選ばれず、逆に選ばれていなかったものが含まれる場合がある)。セル末尾で、これら4遺伝子セットについて (a) 対応するMSigDBコレクションに実在すること、(b) fgsea(minSize=10, maxSize=500)を通過し実際に検定済みであること、の両方を確認するプログラム上の存在チェックを行い、いずれかを満たさない場合は `stopifnot` で処理を停止する。
>
> Methods記載案(v8): *To characterize category-level evidence without an arbitrary per-category cutoff, all Jaccard-deduplicated candidate gene sets (pairwise Jaccard similarity on gene-set membership >= 0.8 collapsed to the lower-padj representative) within each thematic category (IFN, OXPHOS, Proteostasis) were tallied for the number reaching FDR < 0.05 (primary) and FDR < 0.25 (sensitivity) in the corresponding preranked GSEA, together with the concordance of enrichment direction (sign of NES) among the FDR-significant gene sets. For the subsequent per-pathway nonlinear age-trajectory analyses (breakpoint regression and GAM with bootstrap resampling), a single representative gene set per category (two for Proteostasis) was pre-specified independently of, and prior to inspecting, these FDR/ranking results — HALLMARK_INTERFERON_ALPHA_RESPONSE (IFN), HALLMARK_OXIDATIVE_PHOSPHORYLATION (OXPHOS), and REACTOME_AUTOPHAGY plus KEGG_MEDICUS_REFERENCE_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION (Proteostasis) — rather than selected post hoc on the basis of GSEA significance ranking. Membership of each pre-specified gene set among the fgsea-tested gene sets (i.e., passing minSize = 10, maxSize = 500) was verified programmatically before use.*


In [ ]:
%%R
# --- 16-B: IFN / OXPHOS / Proteostasis 関連パスウェイをキーワードで抽出 ---
category_keywords <- list(
  IFN = c("INTERFERON", "TYPE_I_INTERFERON", "TYPE_II_INTERFERON"),
  OXPHOS = c("OXIDATIVE_PHOSPHORYLATION", "RESPIRATORY_ELECTRON_TRANSPORT",
             "ELECTRON_TRANSPORT_CHAIN", "MITOCHONDRIAL_RESPIR"),
  Proteostasis = c("UNFOLDED_PROTEIN_RESPONSE", "PROTEIN_FOLDING", "CHAPERONE",
                    "PROTEASOME", "ENDOPLASMIC_RETICULUM_STRESS", "AUTOPHAGY",
                    "PROTEIN_SECRETION", "\\bERAD\\b")
)

all_collections <- list(Hallmark = pathways_hallmark, Reactome = pathways_reactome,
                         KEGG = pathways_kegg, GO_BP = pathways_gobp)

all_gsea_res <- rbind(res_hallmark[, c("pathway", "NES", "pval", "padj")],
                       res_reactome[, c("pathway", "NES", "pval", "padj")],
                       res_kegg[, c("pathway", "NES", "pval", "padj")],
                       res_gobp[, c("pathway", "NES", "pval", "padj")])

# [v6修正] 候補パスウェイの母集団を、実際にpreranked GSEAで検定された
# (= fgseaのminSize/maxSizeを通過し、NES/pval/padjが全てNAでない)遺伝子セットに限定する。
# 修正前は all_collections (msigdbrから取得した全遺伝子セット、フィルタ前) をキーワード検索
# の母集団にしていたため、fgseaでminSize=10未満のため検定されなかった遺伝子セット
# (例: GOBP_MITOCHONDRIAL_RESPIRASOME_ASSEMBLY, n_genes=9)が非線形transition-age解析の
# 対象に紛れ込む不整合があった。ここでは candidate pathway ⊆ GSEA-tested pathway を保証する。
tested_pathways <- all_gsea_res$pathway[
  !is.na(all_gsea_res$NES) & !is.na(all_gsea_res$pval) & !is.na(all_gsea_res$padj)
]
cat("GSEA検定済み(NES/pval/padjが全て非NA)パスウェイ数:", length(unique(tested_pathways)), "\n")

# [v8修正] MAX_PER_CATEGORYによる打ち切りを撤廃した。
# v6/v7では、Jaccard整理後の候補をpadj昇順に並べ替えたうえで各カテゴリ上位
# MAX_PER_CATEGORY(=8)個のみをdownstream解析に投入していたが、
# (a) 「8個」という基準自体の恣意性、(b) 上位に入らなかった候補の黙示的な除外、
# という2点で頑健性への疑義を招きうる。v8では打ち切りロジックを完全に削除し、
# Jaccard整理後の全候補を保持したまま下でcategory_evidence_dfを作成する
# (per-pathway解析(16.2以降)に投入する遺伝子セットは、本セル末尾で
#  事前specifiedした4遺伝子セットに別途限定する。詳細は本セルのMarkdownを参照)。

# 除外パターン: KEGG_MEDICUSの「特定疾患変異体(HTT/ABeta/PrPSc/UBQLN2/SOD1等)の
# 26Sプロテアソーム分解」派生パスウェイ群。これらは一般的なproteostasis機能そのもの
# ではなく、疾患特異的variantの分解過程を記述したKEGG_MEDICUS特有の枝分かれセットであり、
# かつコアの26Sプロテアソームサブユニット遺伝子を共有して内容がほぼ重複するため除外する。
EXCLUDE_PATTERN <- "KEGG_MEDICUS_VARIANT_"

# Jaccard類似度に基づく重複パスウェイの除去(遺伝子セットがほぼ同一のものを1つに統合し、
# 見かけ上の「複数パスウェイでの再現」による疑似的な頑健性の水増しを防ぐ)
JACCARD_THRESHOLD <- 0.8

dedup_by_jaccard <- function(gene_sets, threshold = JACCARD_THRESHOLD) {
  # gene_setsは既にpadj優先順にソート済みという前提。先に出てくるもの(=より有意)を残す。
  keys <- names(gene_sets)
  n <- length(keys)
  keep <- rep(TRUE, n)
  dropped_for <- setNames(vector("list", n), keys)
  if (n <= 1) return(list(keep = keep, dropped_for = dropped_for))
  for (i in seq_len(n - 1)) {
    if (!keep[i]) next
    for (j in seq(i + 1, n)) {
      if (!keep[j]) next
      gi <- gene_sets[[i]]; gj <- gene_sets[[j]]
      jacc <- length(intersect(gi, gj)) / length(union(gi, gj))
      if (!is.na(jacc) && jacc >= threshold) {
        keep[j] <- FALSE
        dropped_for[[i]] <- c(dropped_for[[i]], sprintf("%s (Jaccard=%.2f)", keys[j], jacc))
      }
    }
  }
  list(keep = keep, dropped_for = dropped_for)
}

pathways_candidate <- list()
pathway_category_map_candidate <- character(0)

for (cat_name in names(category_keywords)) {
  pattern <- paste(category_keywords[[cat_name]], collapse = "|")
  cat_hits <- list()
  for (coll_name in names(all_collections)) {
    nm <- names(all_collections[[coll_name]])
    hit_idx <- grepl(pattern, nm, ignore.case = TRUE)
    for (pw in nm[hit_idx]) {
      key <- paste0(coll_name, "::", pw)
      cat_hits[[key]] <- all_collections[[coll_name]][[pw]]
    }
  }
  if (length(cat_hits) == 0) {
    cat(sprintf("[%s] マッチするパスウェイなし\n", cat_name))
    next
  }

  # 0) [v6追加] GSEAで検定されていない(fgseaのminSize/maxSizeを通過しなかった、または
  #    NES/pval/padjのいずれかがNAの)候補を除外する
  bare_names_all <- sub("^.*::", "", names(cat_hits))
  is_untested <- !(bare_names_all %in% tested_pathways)
  if (any(is_untested)) {
    cat(sprintf("[%s] GSEA未検定(minSize/maxSize不適合等)のため%d件除外: %s\n",
                cat_name, sum(is_untested),
                paste(bare_names_all[is_untested], collapse = ", ")))
    cat_hits <- cat_hits[!is_untested]
  }
  if (length(cat_hits) == 0) {
    cat(sprintf("[%s] GSEA検定済み候補なし\n", cat_name))
    next
  }

  # 1) 疾患variant派生パスウェイを除外
  is_excluded <- grepl(EXCLUDE_PATTERN, names(cat_hits))
  if (any(is_excluded)) {
    cat(sprintf("[%s] KEGG_MEDICUS variant派生パスウェイを%d件除外: %s\n",
                cat_name, sum(is_excluded),
                paste(sub("^.*::", "", names(cat_hits)[is_excluded]), collapse = ", ")))
    cat_hits <- cat_hits[!is_excluded]
  }
  if (length(cat_hits) == 0) {
    cat(sprintf("[%s] 除外後にマッチするパスウェイなし\n", cat_name))
    next
  }

  # 2) padj(小さいほど優先)・遺伝子セットサイズで順序付け
  #    (この順序はJaccard重複除去の「どちらを残すか」の決定にのみ使う。
  #     v8ではこの後の「上位N件で打ち切る」処理は行わない)
  bare_names <- sub("^.*::", "", names(cat_hits))
  padj_lookup <- all_gsea_res$padj[match(bare_names, all_gsea_res$pathway)]
  ord <- order(ifelse(is.na(padj_lookup), Inf, padj_lookup), -lengths(cat_hits))
  cat_hits <- cat_hits[ord]

  # 3) Jaccard>=0.8の重複パスウェイを除去(padjが良い方を残す)
  dd <- dedup_by_jaccard(cat_hits)
  if (any(!dd$keep)) {
    for (k in names(dd$dropped_for)) {
      if (length(dd$dropped_for[[k]]) > 0) {
        cat(sprintf("[%s] %s と重複(遺伝子セットがほぼ同一)のため除外: %s\n",
                    cat_name, sub("^.*::", "", k), paste(dd$dropped_for[[k]], collapse = "; ")))
      }
    }
    cat_hits <- cat_hits[dd$keep]
  }

  # [v8] 4) MAX_PER_CATEGORYによる打ち切りは撤廃。Jaccard整理後の全候補をそのまま保持する。
  cat(sprintf("[%s] %d candidate pathways retained (除外・重複整理後、打ち切りなし):\n",
              cat_name, length(cat_hits)))
  print(sub("^.*::", "", names(cat_hits)))

  pathways_candidate[names(cat_hits)] <- cat_hits
  pathway_category_map_candidate[names(cat_hits)] <- cat_name
}

cat("\n合計候補パスウェイ数(Jaccard整理後、打ち切りなし):", length(pathways_candidate), "\n")

# Sanity check(v6から継続): candidate pathway ⊆ GSEA-tested pathway を保証できているか検証
final_bare_names <- sub("^.*::", "", names(pathways_candidate))
stopifnot(all(final_bare_names %in% tested_pathways))
cat("OK: 全候補パスウェイがGSEA検定済み集合に含まれることを確認しました。\n")

# ============================================================
# [v8新設] category_evidence_df: カテゴリ単位のFDRエビデンス集計
# Jaccard整理後の全候補(打ち切りなし)を対象に、PRIMARY_FDR / SENSITIVITY_FDR
# それぞれの閾値での有意パスウェイ数と、有意パスウェイ間のNES符号(方向)一致度を集計する。
# ============================================================
PRIMARY_FDR <- 0.05
SENSITIVITY_FDR <- 0.25

category_evidence_list <- lapply(names(category_keywords), function(cat_name) {
  keys <- names(pathway_category_map_candidate)[pathway_category_map_candidate == cat_name]
  bare <- sub("^.*::", "", keys)
  idx <- match(bare, all_gsea_res$pathway)
  padj_v <- all_gsea_res$padj[idx]
  nes_v  <- all_gsea_res$NES[idx]

  is_sig_primary <- !is.na(padj_v) & padj_v < PRIMARY_FDR
  is_sig_sensitivity <- !is.na(padj_v) & padj_v < SENSITIVITY_FDR

  sign_primary <- sign(nes_v[is_sig_primary])
  sign_sensitivity <- sign(nes_v[is_sig_sensitivity])

  n_pos_primary <- sum(sign_primary > 0)
  n_neg_primary <- sum(sign_primary < 0)
  n_pos_sensitivity <- sum(sign_sensitivity > 0)
  n_neg_sensitivity <- sum(sign_sensitivity < 0)

  data.frame(
    category = cat_name,
    n_candidates_post_jaccard = length(keys),
    n_significant_FDR05 = sum(is_sig_primary),
    n_positive_NES_FDR05 = n_pos_primary,
    n_negative_NES_FDR05 = n_neg_primary,
    direction_agreement_FDR05 = ifelse(sum(is_sig_primary) > 0,
                                        max(n_pos_primary, n_neg_primary) / sum(is_sig_primary),
                                        NA_real_),
    dominant_sign_FDR05 = ifelse(sum(is_sig_primary) == 0, NA_character_,
                                  ifelse(n_pos_primary >= n_neg_primary, "positive", "negative")),
    n_significant_FDR25 = sum(is_sig_sensitivity),
    n_positive_NES_FDR25 = n_pos_sensitivity,
    n_negative_NES_FDR25 = n_neg_sensitivity,
    direction_agreement_FDR25 = ifelse(sum(is_sig_sensitivity) > 0,
                                        max(n_pos_sensitivity, n_neg_sensitivity) / sum(is_sig_sensitivity),
                                        NA_real_),
    dominant_sign_FDR25 = ifelse(sum(is_sig_sensitivity) == 0, NA_character_,
                                  ifelse(n_pos_sensitivity >= n_neg_sensitivity, "positive", "negative")),
    stringsAsFactors = FALSE
  )
})

category_evidence_df <- do.call(rbind, category_evidence_list)
cat("\n=== category_evidence_df(カテゴリ単位のFDRエビデンス集計) ===\n")
print(category_evidence_df)

write.csv(category_evidence_df,
          file.path(out_dir, "OEP001041_CategoryLevelEvidence_FDR.csv"),
          row.names = FALSE)
cat("\n保存しました:", file.path(out_dir, "OEP001041_CategoryLevelEvidence_FDR.csv"), "\n")

# ============================================================
# [v8新設] representative_spec: downstream(16.2以降)のper-pathway解析に
# 投入する代表パスウェイの事前specification(post-hoc選択ではない)
#
# 以下の4遺伝子セットは、上のcategory_evidence_df(FDR/方向一致の集計結果)を
# 確認する前に、各カテゴリの中核的な生物学的プロセスを代表する標準的な
# MSigDB遺伝子セットとして独立に決定したものである。fgseaのpadjランキングで
# 上位に来たから選んだものではない(実際、v6/v7のMAX_PER_CATEGORY打ち切り
# ロジックで選ばれていたパスウェイの一部はここでは選ばれず、逆に選ばれて
# いなかったものが含まれる場合がある)。
# ============================================================
representative_spec <- list(
  IFN = list(list(collection = "Hallmark", pathway = "HALLMARK_INTERFERON_ALPHA_RESPONSE")),
  OXPHOS = list(list(collection = "Hallmark", pathway = "HALLMARK_OXIDATIVE_PHOSPHORYLATION")),
  Proteostasis = list(
    list(collection = "Reactome", pathway = "REACTOME_AUTOPHAGY"),
    list(collection = "KEGG", pathway = "KEGG_MEDICUS_REFERENCE_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION")
  )
)

# 存在チェック: (a) 該当コレクションに遺伝子セットが実在するか、
#               (b) fgseaで実際に検定済み(tested_pathways)か、の両方を確認する。
missing_report <- character(0)
pathways_selected <- list()
pathway_category_map <- character(0)

for (cat_name in names(representative_spec)) {
  for (spec in representative_spec[[cat_name]]) {
    coll <- spec$collection
    pw   <- spec$pathway
    key  <- paste0(coll, "::", pw)

    exists_in_collection <- !is.null(all_collections[[coll]]) &&
      (pw %in% names(all_collections[[coll]]))
    exists_in_tested <- pw %in% tested_pathways

    if (!exists_in_collection) {
      missing_report <- c(missing_report,
        sprintf("[%s] %s: %sコレクションに遺伝子セットが見つかりません", cat_name, pw, coll))
      next
    }
    if (!exists_in_tested) {
      missing_report <- c(missing_report,
        sprintf("[%s] %s: fgseaで検定されていません(minSize/maxSize不適合等)", cat_name, pw))
      next
    }

    pathways_selected[[key]] <- all_collections[[coll]][[pw]]
    pathway_category_map[key] <- cat_name
    cat(sprintf("[%s] 代表パスウェイとして採用: %s (n_genes=%d)\n",
                cat_name, key, length(all_collections[[coll]][[pw]])))
  }
}

if (length(missing_report) > 0) {
  cat("\n--- 代表パスウェイの存在チェックで問題が見つかりました ---\n")
  cat(paste(missing_report, collapse = "\n"), "\n")
}
stopifnot(length(missing_report) == 0)

cat("\n=== v8: downstream解析(16.2以降)に投入する代表パスウェイ(事前specified、計",
    length(pathways_selected), "件) ===\n")
print(data.frame(category = unname(pathway_category_map),
                  pathway_key = names(pathway_category_map),
                  n_genes = sapply(pathways_selected[names(pathway_category_map)], length),
                  row.names = NULL))


In [ ]:
%%R
# --- 16-C: voom log-CPMから遺伝子セットスコア(平均z-score法)を計算 ---
expr <- v$E  # genes(バージョン付きEnsembl ID) x samples, voom正規化log2-CPM
gene_bare <- sub("\\..*", "", rownames(expr))

# 同一Ensembl IDに重複がある場合は平均発現が最大の行を代表として採用
dup_gene <- gene_bare[duplicated(gene_bare) | duplicated(gene_bare, fromLast = TRUE)]
if (length(dup_gene) > 0) {
  keep_idx <- tapply(seq_len(nrow(expr)), gene_bare, function(idx) {
    if (length(idx) == 1) return(idx)
    idx[which.max(rowMeans(expr[idx, , drop = FALSE]))]
  })
  expr <- expr[unlist(keep_idx), ]
  rownames(expr) <- names(keep_idx)
} else {
  rownames(expr) <- gene_bare
}
cat("スコア計算用入力: 遺伝子", nrow(expr), "x サンプル", ncol(expr), "\n")
stopifnot(identical(colnames(expr), meta_sub$sample))

# 遺伝子ごとにサンプル間でz-score化(平均0, SD1)。分散ゼロの遺伝子はNA化して寄与させない。
gene_mean <- rowMeans(expr)
gene_sd   <- apply(expr, 1, sd)
gene_sd[gene_sd == 0] <- NA
expr_z <- (expr - gene_mean) / gene_sd

pathway_zscore <- function(gene_set) {
  present <- intersect(gene_set, rownames(expr_z))
  if (length(present) < 5) return(rep(NA_real_, ncol(expr_z)))
  colMeans(expr_z[present, , drop = FALSE], na.rm = TRUE)
}

pathway_scores <- t(sapply(pathways_selected, pathway_zscore))
colnames(pathway_scores) <- colnames(expr_z)

n_matched <- sapply(pathways_selected, function(g) length(intersect(g, rownames(expr_z))))
cat("パスウェイスコア行列:", nrow(pathway_scores), "pathways x", ncol(pathway_scores), "samples\n")
cat("パスウェイあたりマッチ遺伝子数の範囲:", range(n_matched), "\n")
if (any(n_matched < 5)) {
  cat("警告: マッチ遺伝子数が5未満のパスウェイがあります(スコアはNAになります):\n")
  print(names(n_matched)[n_matched < 5])
}

### 16.2 パスウェイごとの breakpoint 回帰 + GAM 交差検証

`fit_transition_age()` は1パスウェイ分のパスウェイ活性スコア(平均z-score)を受け取り、(1) `segmented` による breakpoint 回帰(初期値を3通り試し、AIC最良のものを採用)、(2) davies検定による非線形性の存在検定、(3) ケースリサンプリングブートストラップによる95%CI、(4) `mgcv::gam` による独立な平滑化フィットと steepest-slope age、を計算して1行のデータフレームとして返します。

In [ ]:
%%R
# --- 16-D: transition age 推定関数 ---
set.seed(42)
N_BOOT <- 200      # segmented breakpoint と GAM の両方に使うcase-resampling回数
                    # (同一のリサンプルidxを両手法で共有し、resample単位で対応させた比較を可能にする)
                    # ※ GAM側も同数の再フィットが加わるため、パスウェイ数によっては実行時間が
                    #   従来のおよそ2倍程度になる点に留意してください。

# --- 境界アーティファクト対策(16-Lのブートストラップ分布診断を受けた修正) ---
# GAMのsteepest-slope age推定(grid[which.max(abs(d1))])は、データが疎になる
# 年齢範囲の両端で不安定になり、40.02歳・69.53歳付近に繰り返し張り付く多峰性を
# 示すことが確認された。この結果、GAM側のブートストラップCIが解析区間のほぼ
# 全域を覆ってしまい、segmentedとの「手法間相互検証」が意味を失う(何と比べても
# CIが重なってしまう)。対策として、GAM側のargmax探索を年齢範囲の下位・上位
# TRIM_FRAC を除いた領域に限定する。TRIM_FRACの妥当性は16-Iのk感度分析、
# および別途 0.05/0.15 等での再実行で確認すること(このノートブックでは
# 0.10 = 40–70歳なら約43–67歳を既定値とする)。
TRIM_FRAC <- 0.10
# 「境界に触れている」と判定する許容誤差(トリミング後の探索窓の端からの距離)。
# グリッド刻み幅(年齢範囲を200等分)のおよそ2ステップ分を許容誤差とする。
BOUNDARY_TOL_GRID_STEPS <- 2

# 平滑化GAMフィットからsteepest-slope age(|d(score)/dAge|が最大となる年齢)を
# 求めるヘルパー関数。探索を [trim_lo, trim_hi] に限定することで端点付近の
# 境界アーティファクトを避け、それでも探索窓の端に張り付いた場合は
# touches_boundary = TRUE でフラグする(トリミングしてもなお不安定という意味)。
gam_steepest_slope_age <- function(gam_fit, grid, gender_levels, trim_lo, trim_hi) {
  ref_gender <- gender_levels[1]
  pred <- predict(gam_fit, newdata = data.frame(Age = grid,
                   Gender = factor(ref_gender, levels = gender_levels)))
  d1 <- diff(pred) / diff(grid)
  grid_d1 <- head(grid, -1)  # 元コードと同じ対応づけ(各d1[i]をgrid[i]に紐づける)
  step <- diff(grid)[1]
  tol <- BOUNDARY_TOL_GRID_STEPS * step

  search_idx <- which(grid_d1 >= trim_lo & grid_d1 <= trim_hi)
  if (length(search_idx) == 0) {
    return(list(age = NA_real_, touches_boundary = NA))
  }
  best_local <- search_idx[which.max(abs(d1[search_idx]))]
  age_est <- grid_d1[best_local]
  touches <- (age_est <= trim_lo + tol) || (age_est >= trim_hi - tol)
  list(age = age_est, touches_boundary = touches)
}

fit_transition_age <- function(pathway_key, scores_row, age, gender) {
  df <- data.frame(score = as.numeric(scores_row), Age = age, Gender = gender)

  # 1) 線形モデル(帰無仮説: 年齢効果は線形)
  lm0 <- lm(score ~ Age + Gender, data = df)

  # 2) breakpoint回帰: 複数の初期値で頑健にフィットし、AIC最良を採用
  psi_starts <- as.numeric(quantile(df$Age, probs = c(0.35, 0.5, 0.65)))
  seg_fits <- lapply(psi_starts, function(p0) {
    tryCatch(segmented(lm0, seg.Z = ~Age, psi = p0), error = function(e) NULL)
  })
  seg_fits <- Filter(Negate(is.null), seg_fits)
  seg_fit <- if (length(seg_fits) > 0) seg_fits[[which.min(sapply(seg_fits, AIC))]] else NULL

  davies_p <- tryCatch(davies.test(lm0, seg.Z = ~Age)$p.value, error = function(e) NA_real_)

  psi_est <- NA_real_; psi_se <- NA_real_; aic_seg <- NA_real_
  slope_pre <- NA_real_; slope_post <- NA_real_
  if (!is.null(seg_fit)) {
    psi_est <- seg_fit$psi[1, "Est."]
    psi_se  <- seg_fit$psi[1, "St.Err"]
    aic_seg <- AIC(seg_fit)
    sl <- slope(seg_fit)$Age[, "Est."]
    slope_pre <- sl[1]; slope_post <- sl[2]
  }

  # 3) GAM(mgcv)による独立の非線形フィット + steepest-slope age(点推定)
  #    境界アーティファクトを避けるため、探索は [trim_lo, trim_hi] に限定する。
  k_use <- min(5, length(unique(df$Age)) - 1)
  gam_fit <- tryCatch(gam(score ~ s(Age, k = k_use) + Gender, data = df, method = "REML"),
                       error = function(e) NULL)
  gam_transition <- NA_real_
  gam_touches_boundary <- NA
  age_range <- range(df$Age)
  trim_lo <- age_range[1] + TRIM_FRAC * diff(age_range)
  trim_hi <- age_range[2] - TRIM_FRAC * diff(age_range)
  if (!is.null(gam_fit)) {
    grid <- seq(age_range[1], age_range[2], length.out = 200)
    res <- gam_steepest_slope_age(gam_fit, grid, levels(df$Gender), trim_lo, trim_hi)
    gam_transition <- res$age
    gam_touches_boundary <- res$touches_boundary
  }

  # 4) ケースリサンプリングブートストラップ: segmented と GAM の安定性を同一リサンプルで評価
  #    GAM側のtrim_lo/trim_hiは元データ(df)から固定した値を全リサンプルで共有する
  #    (リサンプルごとに探索窓が動くと、CI幅がトリミングそのものの副作用を反映して
  #    しまい、推定の安定性を正しく測れなくなるため)。
  n <- nrow(df)
  boot_psi <- rep(NA_real_, N_BOOT)
  boot_gam <- rep(NA_real_, N_BOOT)
  if (!is.null(seg_fit) || !is.null(gam_fit)) {
    for (b in seq_len(N_BOOT)) {
      idx <- sample.int(n, n, replace = TRUE)
      db <- df[idx, ]

      if (!is.null(seg_fit)) {
        boot_psi[b] <- tryCatch({
          m0 <- lm(score ~ Age + Gender, data = db)
          sf <- segmented(m0, seg.Z = ~Age, psi = psi_est)
          sf$psi[1, "Est."]
        }, error = function(e) NA_real_)
      }

      if (!is.null(gam_fit)) {
        boot_gam[b] <- tryCatch({
          k_use_b <- min(k_use, length(unique(db$Age)) - 1)
          gf_b <- gam(score ~ s(Age, k = k_use_b) + Gender, data = db, method = "REML")
          grid_b <- seq(min(db$Age), max(db$Age), length.out = 200)
          # trim_lo/trim_hiは元データ由来の固定値。リサンプル後の範囲がそれより
          # 狭い場合のみその範囲内にクリップする(探索対象が空にならないための保険)。
          trim_lo_b <- max(trim_lo, min(grid_b))
          trim_hi_b <- min(trim_hi, max(grid_b))
          res_b <- gam_steepest_slope_age(gf_b, grid_b, levels(db$Gender), trim_lo_b, trim_hi_b)
          res_b$age
        }, error = function(e) NA_real_)
      }
    }
  }

  boot_ok <- boot_psi[!is.na(boot_psi)]
  ci_boot <- if (length(boot_ok) >= 30) quantile(boot_ok, c(0.025, 0.975)) else c(NA, NA)

  boot_gam_ok <- boot_gam[!is.na(boot_gam)]
  ci_boot_gam <- if (length(boot_gam_ok) >= 30) quantile(boot_gam_ok, c(0.025, 0.975)) else c(NA, NA)

  # 両手法が同一リサンプルに基づくため、resample単位の対応差分(安定性の直接指標)も計算できる
  paired_ok <- !is.na(boot_psi) & !is.na(boot_gam)
  mean_abs_diff_paired <- if (sum(paired_ok) >= 30) mean(abs(boot_psi[paired_ok] - boot_gam[paired_ok])) else NA_real_

  # 境界に触れているかどうかの判定(methods_agreeの再定義で使用、16-F2参照)。
  # - segmented: 元データの年齢範囲そのものの端に触れているか。
  # - GAM: トリミング後の探索窓([trim_lo, trim_hi])の端に触れているか
  #   (トリミングしてもなお端に張り付く場合、境界アーティファクトが残っている証拠)。
  step_full <- diff(age_range) / 200
  tol_full <- BOUNDARY_TOL_GRID_STEPS * step_full
  seg_ci_touches_boundary <- !is.na(ci_boot[1]) && !is.na(ci_boot[2]) &&
    (unname(ci_boot[1]) <= age_range[1] + tol_full || unname(ci_boot[2]) >= age_range[2] - tol_full)
  gam_ci_touches_boundary <- !is.na(ci_boot_gam[1]) && !is.na(ci_boot_gam[2]) &&
    (unname(ci_boot_gam[1]) <= trim_lo + tol_full || unname(ci_boot_gam[2]) >= trim_hi - tol_full)

  data.frame(
    pathway_key = pathway_key,
    category = pathway_category_map[[pathway_key]],
    collection = sub("::.*", "", pathway_key),
    pathway = sub("^.*::", "", pathway_key),
    n_genes = length(pathways_selected[[pathway_key]]),
    davies_p = davies_p,
    aic_linear = AIC(lm0),
    aic_segmented = aic_seg,
    delta_aic = AIC(lm0) - aic_seg,
    transition_age_segmented = psi_est,
    psi_se_analytic = psi_se,
    ci_lo_boot = unname(ci_boot[1]),
    ci_hi_boot = unname(ci_boot[2]),
    n_boot_converged = length(boot_ok),
    slope_before = slope_pre,
    slope_after = slope_post,
    transition_age_gam = gam_transition,
    gam_trim_lo = trim_lo,
    gam_trim_hi = trim_hi,
    gam_point_touches_boundary = gam_touches_boundary,
    ci_lo_boot_gam = unname(ci_boot_gam[1]),
    ci_hi_boot_gam = unname(ci_boot_gam[2]),
    n_boot_converged_gam = length(boot_gam_ok),
    seg_ci_touches_boundary = seg_ci_touches_boundary,
    gam_ci_touches_boundary = gam_ci_touches_boundary,
    mean_abs_diff_paired_boot = mean_abs_diff_paired,
    stringsAsFactors = FALSE
  )
}

In [ ]:
%%R
# --- 16-E: 全選択パスウェイに適用 ---
results_list <- vector("list", length(pathways_selected))
names(results_list) <- names(pathways_selected)

for (i in seq_along(pathways_selected)) {
  pw_key <- names(pathways_selected)[i]
  cat(sprintf("[%d/%d] %s\n", i, length(pathways_selected), pw_key))
  results_list[[pw_key]] <- tryCatch(
    fit_transition_age(pw_key, pathway_scores[pw_key, ], meta_sub$Age, meta_sub$Gender),
    error = function(e) {
      cat("  失敗:", conditionMessage(e), "\n")
      NULL
    }
  )
}

transition_age_df <- do.call(rbind, Filter(Negate(is.null), results_list))
transition_age_df <- transition_age_df[order(transition_age_df$category, transition_age_df$davies_p), ]
rownames(transition_age_df) <- NULL

options(digits = 4)
print(transition_age_df[, c("category", "pathway", "davies_p", "delta_aic",
                             "transition_age_segmented", "ci_lo_boot", "ci_hi_boot",
                             "transition_age_gam", "ci_lo_boot_gam", "ci_hi_boot_gam")])

In [ ]:
%%R
# --- 16-F: カテゴリ単位の transition age サマリー(多重検定補正込み) ---
# [v8] davies検定のp値は、事前specifiedした代表パスウェイ(IFN x 1, OXPHOS x 1,
# Proteostasis x 2 の計4パスウェイ、16-B参照)にわたる複数回の検定であるため、
# BH(Benjamini-Hochberg)法でFDR補正したq値も併記する(打ち切りパラメータ
# MAX_PER_CATEGORYはv8で撤廃済み。カテゴリ単位のFDRエビデンス集計は
# 16-Bのcategory_evidence_df / OEP001041_CategoryLevelEvidence_FDR.csv を参照)。
transition_age_df$davies_padj_BH <- p.adjust(transition_age_df$davies_p, method = "BH")

cat("=== davies検定 p値 vs BH補正後 q値(全", nrow(transition_age_df), "パスウェイ中、p昇順 上位10件) ===\n")
print(head(transition_age_df[order(transition_age_df$davies_p),
      c("category", "pathway", "davies_p", "davies_padj_BH")], 10))

n_sig_raw <- sum(transition_age_df$davies_p < 0.10, na.rm = TRUE)
n_sig_bh  <- sum(transition_age_df$davies_padj_BH < 0.10, na.rm = TRUE)
cat(sprintf("\n補正前 p<0.10: %d パスウェイ / BH補正後 q<0.10: %d パスウェイ\n", n_sig_raw, n_sig_bh))

# 補正後に生き残るパスウェイのみを「頑健」として集約する(本来のロバスト基準)
robust <- subset(transition_age_df, !is.na(transition_age_segmented) & davies_padj_BH < 0.10 & n_boot_converged >= 30)

if (nrow(robust) > 0) {
  summary_by_cat <- aggregate(
    cbind(transition_age_segmented, transition_age_gam) ~ category,
    data = robust,
    FUN = function(x) c(median = median(x), q25 = unname(quantile(x, .25)), q75 = unname(quantile(x, .75)))
  )
  cat("\n=== カテゴリ別 transition age (BH補正後 q<0.10 のパスウェイ n =", nrow(robust), "で集約) ===\n")
  print(summary_by_cat)
} else {
  cat("\n注意: 多重検定補正(BH)後、q<0.10を満たすパスウェイはありませんでした。\n",
      "したがって『IFN/OXPHOS/proteostasisでtransition ageを検出した』と確証的に結論することはできません。\n",
      "以下は補正前の探索的スクリーニング(p<0.10)による参考集計であり、確証された知見ではなく、\n",
      "独立コホートでの再現確認が必要な仮説生成的な結果として扱ってください。\n\n")
  robust_exploratory <- subset(transition_age_df, !is.na(transition_age_segmented) & davies_p < 0.10 & n_boot_converged >= 30)
  if (nrow(robust_exploratory) > 0) {
    summary_by_cat_exploratory <- aggregate(
      cbind(transition_age_segmented, transition_age_gam) ~ category,
      data = robust_exploratory,
      FUN = function(x) c(median = median(x), q25 = unname(quantile(x, .25)), q75 = unname(quantile(x, .75)))
    )
    cat("=== [探索的・補正前] カテゴリ別 transition age (p<0.10 のパスウェイ n =", nrow(robust_exploratory), "で集約) ===\n")
    print(summary_by_cat_exploratory)
  } else {
    cat("補正前の閾値(p<0.10)ですら該当パスウェイがありませんでした。\n")
  }
}

cat("\n--- 全パスウェイ内訳(davies_p 昇順、上位15件、BH補正q値付き) ---\n")
print(head(transition_age_df[order(transition_age_df$davies_p),
      c("category", "pathway", "davies_p", "davies_padj_BH", "transition_age_segmented")], 15))

### 16.2b segmentedとGAMの手法間一致・ブートストラップ安定性の評価(全パスウェイ対象)

IFN以外のカテゴリも含め、選択された全パスウェイについて、(1) segmented breakpointの
ブートストラップ95%CI、(2) GAMのsteepest-slope ageのブートストラップ95%CI(同一のリサンプル
を使用した対応比較)、(3) 両CIの重なり、を評価する。BH補正後に有意(q<0.10)であっても
両手法のCIが重ならない場合は「頑健に検出された」とは言えず、モデル依存の不安定な推定として
扱う。

In [ ]:
%%R
# --- 16-F2: 手法間一致・ブートストラップ安定性の評価(全パスウェイ) ---
transition_age_df$width_ci_segmented <- transition_age_df$ci_hi_boot - transition_age_df$ci_lo_boot
transition_age_df$width_ci_gam <- transition_age_df$ci_hi_boot_gam - transition_age_df$ci_lo_boot_gam

transition_age_df$ci_overlap <- pmax(
  0,
  pmin(transition_age_df$ci_hi_boot, transition_age_df$ci_hi_boot_gam) -
  pmax(transition_age_df$ci_lo_boot, transition_age_df$ci_lo_boot_gam)
)

# methods_agree の再定義。
# 従来はCI重複のみで判定していたが、GAM側のCIが境界アーティファクトにより
# 解析区間のほぼ全域を覆う幅を持つ場合、CI重複は「手法間で推定値が一致した」
# ことを意味せず、「GAM側のCIが機能不全なほど広いので、何と比べても重なって
# しまう」だけになる(16-Lのブートストラップ分布診断を参照)。
# そこで、CI重複に加えて「両CIとも境界に触れていない」ことを必須条件とする:
#   - seg_ci_touches_boundary: segmented CIが元の年齢範囲の端に触れているか
#   - gam_ci_touches_boundary: GAM CIがトリミング後の探索窓([trim_lo, trim_hi])
#     の端に触れているか(16-Dのfit_transition_age()内で判定済み)
transition_age_df$methods_agree <- !is.na(transition_age_df$ci_overlap) &
  transition_age_df$ci_overlap > 0 &
  !transition_age_df$seg_ci_touches_boundary &
  !transition_age_df$gam_ci_touches_boundary

cat("=== 全パスウェイ: FDR補正 + 手法間一致(境界チェック込み) + ブートストラップ安定性 ===\n")
print(transition_age_df[order(transition_age_df$category, transition_age_df$davies_p),
      c("category", "pathway", "davies_p", "davies_padj_BH",
        "transition_age_segmented", "width_ci_segmented", "seg_ci_touches_boundary",
        "transition_age_gam", "width_ci_gam", "gam_ci_touches_boundary",
        "mean_abs_diff_paired_boot", "methods_agree")])

n_bh_sig <- sum(transition_age_df$davies_padj_BH < 0.10, na.rm = TRUE)
n_bh_sig_and_agree <- sum(transition_age_df$davies_padj_BH < 0.10 & transition_age_df$methods_agree, na.rm = TRUE)
n_agree_old_def <- sum(!is.na(transition_age_df$ci_overlap) & transition_age_df$ci_overlap > 0, na.rm = TRUE)
n_agree_new_def <- sum(transition_age_df$methods_agree, na.rm = TRUE)
cat(sprintf("\nBH補正後 q<0.10: %d パスウェイ / うち手法間一致(新定義): %d パスウェイ\n",
            n_bh_sig, n_bh_sig_and_agree))
cat(sprintf("参考: 旧定義(CI重複のみ)で一致と判定された数 = %d / 新定義(CI重複 かつ 両CIとも境界非接触)で一致と判定された数 = %d\n",
            n_agree_old_def, n_agree_new_def))
cat("→ 『BH有意 かつ 手法間一致』の集合のみを、頑健に支持されたtransition ageの候補として扱うのが妥当です。\n")

# 診断(16-I, 16-J)の対象パスウェイを決定:
# 優先度1: BH補正後 q<0.10 のパスウェイ
# 優先度2(フォールバック): 該当なしの場合、カテゴリごとにdavies_pが最小の1パスウェイ(探索的扱いと明記)
target_keys <- transition_age_df$pathway_key[!is.na(transition_age_df$davies_padj_BH) &
                                               transition_age_df$davies_padj_BH < 0.10]
fallback_used <- FALSE
if (length(target_keys) == 0) {
  fallback_used <- TRUE
  cat("\n注意: BH補正後に有意なパスウェイがないため、カテゴリごとにdavies_pが最小の1件を",
      "探索的診断対象とします(確証的な結果ではありません)。\n")
  per_cat_best <- do.call(rbind, lapply(split(transition_age_df, transition_age_df$category),
                                          function(d) d[which.min(d$davies_p), ]))
  target_keys <- per_cat_best$pathway_key
}
cat("\n診断対象パスウェイ(", length(target_keys), "件):\n", sep = "")
print(transition_age_df[transition_age_df$pathway_key %in% target_keys,
      c("category", "pathway", "davies_p", "davies_padj_BH")])

In [ ]:
%%R
# --- 16-G: 可視化(カテゴリごとに最も非線形性の強いパスウェイ) ---
plot_df <- do.call(rbind, lapply(names(pathways_selected), function(pw_key) {
  data.frame(
    pathway_key = pw_key,
    pathway = sub("^.*::", "", pw_key),
    category = pathway_category_map[[pw_key]],
    Age = meta_sub$Age,
    Gender = meta_sub$Gender,
    score = as.numeric(pathway_scores[pw_key, ])
  )
}))

# カテゴリごとにdavies_pが最小(非線形性の証拠が最も強い)パスウェイを1つずつ選択
top_per_cat <- do.call(rbind, lapply(split(transition_age_df, transition_age_df$category), function(d) {
  d[which.min(d$davies_p), ]
}))

plot_df_top <- plot_df[plot_df$pathway_key %in% top_per_cat$pathway_key, ]
vline_df <- top_per_cat[, c("pathway_key", "pathway", "transition_age_segmented", "ci_lo_boot", "ci_hi_boot")]
plot_df_top$pathway <- factor(plot_df_top$pathway, levels = top_per_cat$pathway)
vline_df$pathway <- factor(vline_df$pathway, levels = top_per_cat$pathway)

p <- ggplot(plot_df_top, aes(x = Age, y = score)) +
  geom_rect(data = vline_df, inherit.aes = FALSE,
            aes(xmin = ci_lo_boot, xmax = ci_hi_boot, ymin = -Inf, ymax = Inf),
            fill = "grey70", alpha = 0.3) +
  geom_point(aes(color = Gender), alpha = 0.6, size = 1.5) +
  geom_smooth(method = "gam", formula = y ~ s(x, k = 5), color = "black", se = TRUE) +
  geom_vline(data = vline_df, aes(xintercept = transition_age_segmented),
             linetype = "dashed", color = "firebrick") +
  facet_wrap(~ pathway, scales = "free_y", ncol = 2) +
  labs(x = "Age (years)", y = "ssGSEA score",
       title = "カテゴリ別・最も非線形性の強いパスウェイの年齢軌跡",
       subtitle = "破線 = 推定transition age(segmented)、灰色帯 = ブートストラップ95%CI") +
  theme_bw(base_size = 11)

print(p)

### 16.3 segmented と GAM の乖離の診断(全カテゴリ、16-F2で決定した対象パスウェイ)

16-F2で選定した対象パスウェイ(BH補正後 有意なもの、なければカテゴリ最良候補)について、
(1) GAMの基底次元 `k` を変えた際の steepest-slope age の安定性、(2) 両手法のフィットを
実データに重ねた可視化、で乖離の原因を診断する。IFNに限らず、選定された全カテゴリの
パスウェイを対象とする。

In [ ]:
%%R
# --- 16-I: GAM の基底次元 k に対する steepest-slope age の感度分析 ---
# (target_keys は16-F2で決定済み: BH補正後有意なパスウェイ、フォールバック時はカテゴリ最良候補)
# 境界アーティファクト対策として、fit_transition_age()(16-D)と同じ
# trim_lo/trim_hi(TRIM_FRACで年齢範囲の両端を除外)でargmax探索を制限する。
k_grid <- c(4, 5, 6, 7, 8)

gam_sensitivity <- do.call(rbind, lapply(target_keys, function(pw_key) {
  df <- data.frame(score = as.numeric(pathway_scores[pw_key, ]),
                    Age = meta_sub$Age, Gender = meta_sub$Gender)
  age_range <- range(df$Age)
  trim_lo <- age_range[1] + TRIM_FRAC * diff(age_range)
  trim_hi <- age_range[2] - TRIM_FRAC * diff(age_range)
  do.call(rbind, lapply(k_grid, function(k) {
    gf <- tryCatch(gam(score ~ s(Age, k = k) + Gender, data = df, method = "REML"),
                    error = function(e) NULL)
    if (is.null(gf)) {
      return(data.frame(pathway_key = pw_key, k = k, transition_age = NA_real_,
                         touches_boundary = NA, edf = NA_real_, aic = NA_real_))
    }
    grid <- seq(age_range[1], age_range[2], length.out = 200)
    res <- gam_steepest_slope_age(gf, grid, levels(df$Gender), trim_lo, trim_hi)
    data.frame(pathway_key = pw_key, k = k,
               transition_age = res$age,
               touches_boundary = res$touches_boundary,
               edf = sum(gf$edf) - 1,
               aic = AIC(gf))
  }))
}))
gam_sensitivity$category <- pathway_category_map[gam_sensitivity$pathway_key]
gam_sensitivity$pathway <- sub("^.*::", "", gam_sensitivity$pathway_key)

cat("=== GAM steepest-slope age の k(基底次元)感度分析(全対象パスウェイ、境界トリム後) ===\n")
print(gam_sensitivity[, c("category", "pathway", "k", "transition_age", "touches_boundary", "edf", "aic")])
cat("\n※ transition_ageがkによって大きく動く、あるいはtouches_boundary=TRUEが多いパスウェイほど、\n",
    "   GAM推定は基底次元・境界に依存しており、segmentedとの乖離は『相互検証不一致』というより\n",
    "   『GAM推定自体が不安定』という説明が妥当です。touches_boundary=TRUEの推定値は、\n",
    "   トリミング後もなお探索窓の端に張り付いていることを示すため、信頼できないものとして扱ってください。\n")

In [ ]:
%%R
# --- 16-J: 対象パスウェイの可視化(segmented breakpoint と GAM smooth、両者のブートストラップCIを重ねて表示) ---
plot_df_target <- do.call(rbind, lapply(target_keys, function(pw_key) {
  data.frame(pathway_key = pw_key,
             category = pathway_category_map[[pw_key]],
             pathway = sub("^.*::", "", pw_key),
             Age = meta_sub$Age, Gender = meta_sub$Gender,
             score = as.numeric(pathway_scores[pw_key, ]))
}))

vline_df_target <- transition_age_df[transition_age_df$pathway_key %in% target_keys,
                                       c("pathway_key", "category", "pathway",
                                         "transition_age_segmented", "ci_lo_boot", "ci_hi_boot",
                                         "transition_age_gam", "ci_lo_boot_gam", "ci_hi_boot_gam",
                                         "methods_agree")]

facet_label <- paste0(vline_df_target$category, ": ", vline_df_target$pathway)
plot_df_target$facet_label <- paste0(plot_df_target$category, ": ", plot_df_target$pathway)
plot_df_target$facet_label <- factor(plot_df_target$facet_label, levels = facet_label)
vline_df_target$facet_label <- factor(facet_label, levels = facet_label)

p_target <- ggplot(plot_df_target, aes(x = Age, y = score)) +
  geom_rect(data = vline_df_target, inherit.aes = FALSE,
            aes(xmin = ci_lo_boot, xmax = ci_hi_boot, ymin = -Inf, ymax = Inf),
            fill = "grey60", alpha = 0.3) +
  geom_rect(data = vline_df_target, inherit.aes = FALSE,
            aes(xmin = ci_lo_boot_gam, xmax = ci_hi_boot_gam, ymin = -Inf, ymax = Inf),
            fill = "lightgreen", alpha = 0.3) +
  geom_point(aes(color = Gender), alpha = 0.6, size = 1.5) +
  geom_smooth(method = "gam", formula = y ~ s(x, k = 5), color = "steelblue", se = TRUE) +
  geom_vline(data = vline_df_target, aes(xintercept = transition_age_segmented),
             linetype = "dashed", color = "firebrick") +
  geom_vline(data = vline_df_target, aes(xintercept = transition_age_gam),
             linetype = "dotted", color = "darkgreen") +
  facet_wrap(~ facet_label, scales = "free_y", ncol = 2) +
  labs(x = "Age (years)", y = "Pathway z-score",
       title = "対象パスウェイ: segmented(赤破線・灰帯) vs GAM(緑点線・緑帯)のtransition age比較",
       subtitle = "帯はそれぞれのブートストラップ95%CI。重なりがない場合は手法間で不一致(methods_agree=FALSE)。") +
  theme_bw(base_size = 11)

print(p_target)
ggsave(file.path(out_dir, "OEP001041_segmentedVsGAM_diagnostic_allCategories.png"),
       plot = p_target, width = 9, height = 8, dpi = 150)

### 16.4 IFN深掘り診断: GAMのk依存性(k=5–8) + ブートストラップtransition age分布

16.3の診断をIFNパスウェイに絞ってさらに掘り下げる。(1) GAMの平滑化曲線そのものを
k=5〜8で重ねて描画し、変曲点周辺の形状がkでどれだけ変わるかを目視確認する。
(2) segmentedとGAMそれぞれのケースリサンプリングブートストラップから得られる
transition ageの**分布そのもの**(点推定・CIの要約だけでなく)を可視化し、
Kolmogorov-Smirnov検定で2つの分布が統計的に異なるかどうかを確認する。

In [ ]:
%%R
# --- 16-K: IFNパスウェイのGAM曲線(k=5–8)重ね合わせ ---
ifn_keys <- transition_age_df$pathway_key[transition_age_df$category == "IFN" & transition_age_df$davies_p < 0.10]
if (length(ifn_keys) == 0) {
  cat("davies_p<0.10のIFNパスウェイがないため、davies_pが最小の1件を対象とします(探索的)。\n")
  ifn_rows <- transition_age_df[transition_age_df$category == "IFN", ]
  ifn_keys <- ifn_rows$pathway_key[which.min(ifn_rows$davies_p)]
}
cat("対象IFNパスウェイ:", paste(sub("^.*::", "", ifn_keys), collapse = ", "), "\n")

k_range <- 5:8

gam_curves_ifn <- do.call(rbind, lapply(ifn_keys, function(pw_key) {
  df <- data.frame(score = as.numeric(pathway_scores[pw_key, ]), Age = meta_sub$Age, Gender = meta_sub$Gender)
  do.call(rbind, lapply(k_range, function(k) {
    gf <- tryCatch(gam(score ~ s(Age, k = k) + Gender, data = df, method = "REML"), error = function(e) NULL)
    if (is.null(gf)) return(NULL)
    grid <- seq(min(df$Age), max(df$Age), length.out = 200)
    ref_gender <- levels(df$Gender)[1]
    pred <- predict(gf, newdata = data.frame(Age = grid, Gender = factor(ref_gender, levels = levels(df$Gender))),
                     se.fit = TRUE)
    data.frame(pathway_key = pw_key, k = factor(k), Age = grid, fit = pred$fit, se = pred$se.fit)
  }))
}))
gam_curves_ifn$pathway <- sub("^.*::", "", gam_curves_ifn$pathway_key)

points_df_ifn <- do.call(rbind, lapply(ifn_keys, function(pw_key) {
  data.frame(pathway_key = pw_key, pathway = sub("^.*::", "", pw_key),
             Age = meta_sub$Age, score = as.numeric(pathway_scores[pw_key, ]))
}))

p_gamk <- ggplot() +
  geom_point(data = points_df_ifn, aes(x = Age, y = score), color = "grey40", alpha = 0.35, size = 1.3) +
  geom_ribbon(data = gam_curves_ifn, aes(x = Age, ymin = fit - 1.96 * se, ymax = fit + 1.96 * se, fill = k),
              alpha = 0.12) +
  geom_line(data = gam_curves_ifn, aes(x = Age, y = fit, color = k), linewidth = 0.9) +
  facet_wrap(~ pathway, scales = "free_y", ncol = 1) +
  labs(x = "Age (years)", y = "Pathway z-score", color = "GAM k", fill = "GAM k",
       title = "IFNパスウェイ: GAM平滑化曲線のk依存性(k=5–8)",
       subtitle = "kによって曲線形状・急勾配点の位置がどれだけ変わるかを目視確認") +
  theme_bw(base_size = 11)

print(p_gamk)
ggsave(file.path(out_dir, "OEP001041_IFN_GAM_k_curves.png"), plot = p_gamk, width = 7, height = 8, dpi = 150)

In [ ]:
%%R
# --- 16-L: IFNパスウェイのブートストラップtransition age分布(segmented vs GAM、境界トリム後) ---
# 16-Dのブートストラップは要約CIのみを保持するため、ここでは同じ手続きを再実行し、
# 生の200回分の推定値そのものを分布として可視化する(乱数シードは16-Dと同じ42を使用)。
# GAM側は fit_transition_age()(16-D)と同じtrim_lo/trim_hi(TRIM_FRAC)で
# argmax探索を制限する(境界アーティファクト対策の効果をここで直接確認できる)。
N_BOOT_DIAG <- 200
set.seed(42)

bootstrap_transition_distributions <- function(pw_key) {
  df <- data.frame(score = as.numeric(pathway_scores[pw_key, ]), Age = meta_sub$Age, Gender = meta_sub$Gender)
  lm0 <- lm(score ~ Age + Gender, data = df)
  psi_start <- as.numeric(quantile(df$Age, 0.5))
  seg_fit0 <- tryCatch(segmented(lm0, seg.Z = ~Age, psi = psi_start), error = function(e) NULL)
  psi_est <- if (!is.null(seg_fit0)) seg_fit0$psi[1, "Est."] else psi_start
  k_use <- min(5, length(unique(df$Age)) - 1)
  age_range <- range(df$Age)
  trim_lo <- age_range[1] + TRIM_FRAC * diff(age_range)
  trim_hi <- age_range[2] - TRIM_FRAC * diff(age_range)

  n <- nrow(df)
  boot_psi <- rep(NA_real_, N_BOOT_DIAG)
  boot_gam <- rep(NA_real_, N_BOOT_DIAG)
  for (b in seq_len(N_BOOT_DIAG)) {
    idx <- sample.int(n, n, replace = TRUE)
    db <- df[idx, ]
    boot_psi[b] <- tryCatch({
      m0 <- lm(score ~ Age + Gender, data = db)
      sf <- segmented(m0, seg.Z = ~Age, psi = psi_est)
      sf$psi[1, "Est."]
    }, error = function(e) NA_real_)
    boot_gam[b] <- tryCatch({
      k_use_b <- min(k_use, length(unique(db$Age)) - 1)
      gf_b <- gam(score ~ s(Age, k = k_use_b) + Gender, data = db, method = "REML")
      grid_b <- seq(min(db$Age), max(db$Age), length.out = 200)
      trim_lo_b <- max(trim_lo, min(grid_b))
      trim_hi_b <- min(trim_hi, max(grid_b))
      res_b <- gam_steepest_slope_age(gf_b, grid_b, levels(db$Gender), trim_lo_b, trim_hi_b)
      res_b$age
    }, error = function(e) NA_real_)
  }
  data.frame(pathway_key = pw_key, boot_id = seq_len(N_BOOT_DIAG), segmented = boot_psi, gam = boot_gam)
}

boot_dist_ifn <- do.call(rbind, lapply(ifn_keys, bootstrap_transition_distributions))

boot_dist_long <- rbind(
  data.frame(pathway_key = boot_dist_ifn$pathway_key, method = "segmented",
             transition_age = boot_dist_ifn$segmented),
  data.frame(pathway_key = boot_dist_ifn$pathway_key, method = "gam",
             transition_age = boot_dist_ifn$gam)
)
boot_dist_long$pathway <- sub("^.*::", "", boot_dist_long$pathway_key)
boot_dist_long <- boot_dist_long[!is.na(boot_dist_long$transition_age), ]

median_df <- aggregate(transition_age ~ pathway + method, data = boot_dist_long, FUN = median)

p_bootdist <- ggplot(boot_dist_long, aes(x = transition_age, fill = method)) +
  geom_density(alpha = 0.45, color = NA) +
  geom_vline(data = median_df, aes(xintercept = transition_age, color = method),
             linetype = "dashed", linewidth = 0.8) +
  facet_wrap(~ pathway, scales = "free", ncol = 1) +
  scale_fill_manual(values = c(segmented = "firebrick", gam = "darkgreen")) +
  scale_color_manual(values = c(segmented = "firebrick", gam = "darkgreen")) +
  labs(x = "Bootstrap transition age (years)", y = "Density",
       title = "IFN pathway: bootstrap distribution of transition age (segmented vs GAM, after boundary trimming)",
       subtitle = sprintf("%d case-resampling bootstrap replicates each. Dashed line = distribution median", N_BOOT_DIAG)) +
  theme_bw(base_size = 11)

print(p_bootdist)
ggsave(file.path(out_dir, "OEP001041_IFN_bootstrap_transitionAge_distributions_fixed.png"),
       plot = p_bootdist, width = 7, height = 8, dpi = 300)

cat("=== ブートストラップ分布の要約(IFN、境界トリム後) ===\n")
summary_tbl <- do.call(rbind, lapply(split(boot_dist_long, list(boot_dist_long$pathway, boot_dist_long$method)), function(d) {
  if (nrow(d) == 0) return(NULL)
  data.frame(pathway = d$pathway[1], method = d$method[1],
             median = median(d$transition_age), IQR = IQR(d$transition_age), sd = sd(d$transition_age))
}))
print(summary_tbl[order(summary_tbl$pathway, summary_tbl$method), ])

cat("\n=== Kolmogorov-Smirnov検定: segmented分布 vs GAM分布が同一かどうか ===\n")
ks_summary <- do.call(rbind, lapply(ifn_keys, function(pw_key) {
  seg_v <- boot_dist_ifn$segmented[boot_dist_ifn$pathway_key == pw_key]
  gam_v <- boot_dist_ifn$gam[boot_dist_ifn$pathway_key == pw_key]
  seg_v <- seg_v[!is.na(seg_v)]; gam_v <- gam_v[!is.na(gam_v)]
  ks_p <- if (length(seg_v) >= 10 && length(gam_v) >= 10) suppressWarnings(ks.test(seg_v, gam_v)$p.value) else NA_real_
  data.frame(pathway = sub("^.*::", "", pw_key),
             median_segmented = median(seg_v), median_gam = median(gam_v),
             ks_test_p = ks_p)
}))
print(ks_summary)
cat("(ks_test_pが小さいほど、segmentedとGAMのブートストラップ分布が統計的に異なることを示唆。\n",
    " 境界トリム後もなお分布が大きく異なる/多峰性が残る場合、GAMのsteepest-slope age推定は\n",
    " このデータには適さない手法であるという結論を補強する。)\n")


In [ ]:
%%R
# --- 16-H: 結果の保存 ---
write.csv(transition_age_df, file.path(out_dir, "OEP001041_PathwayTransitionAge_40to70.csv"), row.names = FALSE)
write.csv(gam_sensitivity, file.path(out_dir, "OEP001041_GAM_k_sensitivity_IFN.csv"), row.names = FALSE)
write.csv(boot_dist_long, file.path(out_dir, "OEP001041_IFN_bootstrap_transitionAge_raw.csv"), row.names = FALSE)
ggsave(file.path(out_dir, "OEP001041_PathwayTransitionAge_topByCategory.png"), plot = p, width = 9, height = 7, dpi = 150)

cat("保存先:", out_dir, "\n")
cat("segmented version:", as.character(packageVersion("segmented")),
    " mgcv version:", as.character(packageVersion("mgcv")), "\n")

### 16.5 [Supplementary/Exploratory] transition age と HRV(α1/α2)ピーク年齢の比較

**重要**: 16.2/16.3/16.4のいずれの解析でも、事前指定した4代表パスウェイ(IFN, OXPHOS, Proteostasis×2)のうち
BH補正後にq<0.10を満たすものはありませんでした(16-F参照)。したがって以下の比較は、
**確証的な主解析ではなく、あくまで探索的・仮説生成的な補足解析**として提示します。

Main Figure(Figure 1: CHI/α1/α2ピーク年齢)とは明確に分離し、本セクションの出力は
**Supplementary Figure**としてのみ本文に組み込むこと。図注・本文中では必ず
"No pathway met the prespecified BH-adjusted significance threshold; transition-age estimates
are exploratory and imprecise." の趣旨を明記する。

In [ ]:
%%R
# --- 16-I: [Supplementary/Exploratory] pathway transition age と HRVピーク年齢(alpha1=57.3y, alpha2=67.1y)の比較 ---
# 注意: 本セルはMain Figureを一切変更しない。出力は全て "Supplementary" 接頭辞のファイルとして
# 別途保存し、本文のFigure 1(CHI/alpha1/alpha2)とは独立に扱うこと。

HRV_ALPHA1_PEAK <- 57.3
HRV_ALPHA2_PEAK <- 67.1

# 16-Fで確認済みの通り、BH補正後にq<0.10を満たすパスウェイは存在しない。
# 図には全パスウェイを含めるが、色/形でBH有意性の有無を明示し、
# 「未補正p<0.10」のパスウェイのみを一応の探索的シグナルとして別枠で強調する。
plot_df <- transition_age_df
plot_df$bh_status <- ifelse(!is.na(plot_df$davies_padj_BH) & plot_df$davies_padj_BH < 0.10,
                             "BH q<0.10", "BH q>=0.10 (not significant)")
stopifnot(all(plot_df$bh_status == "BH q>=0.10 (not significant)"))  # 16-Fの前提を明示的に再確認
plot_df$pathway_label <- sub("^.*::", "", plot_df$pathway)

# segmented法・GAM法の両方をロングフォーマットにまとめる
long_df <- rbind(
  data.frame(pathway_label = plot_df$pathway_label, category = plot_df$category,
             method = "segmented", est = plot_df$transition_age_segmented,
             ci_lo = plot_df$ci_lo_boot, ci_hi = plot_df$ci_hi_boot,
             davies_p = plot_df$davies_p, davies_padj_BH = plot_df$davies_padj_BH),
  data.frame(pathway_label = plot_df$pathway_label, category = plot_df$category,
             method = "GAM (steepest slope)", est = plot_df$transition_age_gam,
             ci_lo = plot_df$ci_lo_boot_gam, ci_hi = plot_df$ci_hi_boot_gam,
             davies_p = plot_df$davies_p, davies_padj_BH = plot_df$davies_padj_BH)
)
long_df <- long_df[!is.na(long_df$est), ]
long_df$exploratory_signal <- ifelse(!is.na(long_df$davies_p) & long_df$davies_p < 0.10,
                                      "uncorrected p<0.10 (exploratory)", "p>=0.10")

CAPTION_NOTE <- paste0(
  "No pathway met the prespecified BH-adjusted significance threshold; ",
  "transition-age estimates are exploratory and imprecise."
)

p_supp <- ggplot(long_df, aes(x = est, y = pathway_label, color = method, shape = exploratory_signal)) +
  geom_vline(xintercept = HRV_ALPHA1_PEAK, linetype = "dashed", color = "steelblue") +
  geom_vline(xintercept = HRV_ALPHA2_PEAK, linetype = "dashed", color = "firebrick") +
  geom_errorbarh(aes(xmin = ci_lo, xmax = ci_hi), height = 0.2,
                 position = position_dodge(width = 0.5)) +
  geom_point(size = 2.3, position = position_dodge(width = 0.5)) +
  facet_grid(category ~ ., scales = "free_y", space = "free_y") +
  scale_color_manual(values = c("segmented" = "darkorange", "GAM (steepest slope)" = "darkgreen")) +
  scale_shape_manual(values = c("uncorrected p<0.10 (exploratory)" = 17, "p>=0.10" = 16)) +
  labs(x = "Estimated transition / steepest-slope age (years)", y = NULL,
       color = "Method", shape = "Uncorrected signal",
       title = "[Supplementary, Exploratory] Pathway transition-age estimates vs. HRV alpha1/alpha2 peaks",
       subtitle = paste0(CAPTION_NOTE, "\n",
         "Dashed lines: HRV alpha1 peak (", HRV_ALPHA1_PEAK, "y, blue) and alpha2 peak (",
         HRV_ALPHA2_PEAK, "y, red). Error bars: bootstrap 95% CI. All estimates: BH q>=0.10.")) +
  theme_bw(base_size = 10) +
  theme(plot.subtitle = element_text(size = 7.5))

print(p_supp)
ggsave(file.path(out_dir, "OEP001041_Supplementary_TransitionAge_vs_HRVpeaks.png"),
       plot = p_supp, width = 9, height = 6, dpi = 150)

# 対応する数値サマリーテーブル(補足資料・本文中の言及に使用)
supp_table <- long_df[order(long_df$category, long_df$pathway_label, long_df$method),
                       c("category", "pathway_label", "method", "est", "ci_lo", "ci_hi",
                         "davies_p", "davies_padj_BH", "exploratory_signal")]
write.csv(supp_table,
          file.path(out_dir, "OEP001041_Supplementary_TransitionAge_vs_HRVpeaks_table.csv"),
          row.names = FALSE)

cat("\n=== [Supplementary, Exploratory] 注記文(本文/図注にそのまま使用可) ===\n")
cat(CAPTION_NOTE, "\n")
cat("\n=== 数値サマリー(全パスウェイ x 2手法) ===\n")
print(supp_table)


### 16.6 血球組成サロゲート補正後の代表パスウェイ transition age 再評価

[v11追加] §16.2-16.3の`fit_transition_age()`と同一のロジックに、§10.5で計算した`NLR_proxy`を共変量として追加した`fit_transition_age_nlr()`を、16-Bで事前specifiedした代表4パスウェイ(`pathways_selected`: IFN, OXPHOS, Proteostasis×2)にのみ適用する。全候補パスウェイではなく代表4件に限定するのは16-Bと同じ計算コスト上の理由による。

> 計算コストの観点から、ここではブートストラップCI(§16.2の`N_BOOT`回リサンプリング)は省略し、davies検定p値・ΔAIC・transition age点推定の比較に限定する。補正後もブートストラップCIまで含めて厳密に確認したい場合は、`fit_transition_age()`と同様のリサンプリングループを本関数にも追加すること。
>
> 解釈の骨子: 未補正(§16.2-16.3の`transition_age_df`)と補正後(`transition_age_df_nlr`)でdavies_p/ΔAICの傾向(離散breakpointを支持しない、という現行の結論を含む)が保たれれば、3.4節の「不連続transitionではなく緩やかなdriftと整合的」という結論はNLR_proxyで捉えられる血球組成シフトに単純に還元されないことを示唆する材料になる。

In [ ]:
%%R
# --- 16-N-1: NLR_proxy補正版のtransition age推定関数 ---
stopifnot("NLR_proxy" %in% colnames(meta_sub))

fit_transition_age_nlr <- function(pathway_key, scores_row, age, gender, nlr) {
  df <- data.frame(score = as.numeric(scores_row), Age = age, Gender = gender, NLR_proxy = nlr)

  # 1) 線形モデル(共変量にNLR_proxyを追加)
  lm0 <- lm(score ~ Age + Gender + NLR_proxy, data = df)

  # 2) breakpoint回帰: 複数の初期値で頑健にフィットし、AIC最良を採用(16-Dと同じロジック)
  psi_starts <- as.numeric(quantile(df$Age, probs = c(0.35, 0.5, 0.65)))
  seg_fits <- lapply(psi_starts, function(p0) {
    tryCatch(segmented(lm0, seg.Z = ~Age, psi = p0), error = function(e) NULL)
  })
  seg_fits <- Filter(Negate(is.null), seg_fits)
  seg_fit <- if (length(seg_fits) > 0) seg_fits[[which.min(sapply(seg_fits, AIC))]] else NULL

  davies_p <- tryCatch(davies.test(lm0, seg.Z = ~Age)$p.value, error = function(e) NA_real_)

  psi_est <- NA_real_; aic_seg <- NA_real_
  if (!is.null(seg_fit)) {
    psi_est <- seg_fit$psi[1, "Est."]
    aic_seg <- AIC(seg_fit)
  }

  # 3) GAM(共変量にNLR_proxyを追加、参照値は中央値で固定)
  k_use <- min(5, length(unique(df$Age)) - 1)
  gam_fit <- tryCatch(gam(score ~ s(Age, k = k_use) + Gender + NLR_proxy, data = df, method = "REML"),
                       error = function(e) NULL)
  gam_transition <- NA_real_
  age_range <- range(df$Age)
  trim_lo <- age_range[1] + TRIM_FRAC * diff(age_range)
  trim_hi <- age_range[2] - TRIM_FRAC * diff(age_range)
  if (!is.null(gam_fit)) {
    grid <- seq(age_range[1], age_range[2], length.out = 200)
    ref_gender <- levels(df$Gender)[1]
    ref_nlr <- median(df$NLR_proxy)
    pred <- predict(gam_fit, newdata = data.frame(Age = grid,
                     Gender = factor(ref_gender, levels = levels(df$Gender)),
                     NLR_proxy = ref_nlr))
    d1 <- diff(pred) / diff(grid)
    grid_d1 <- head(grid, -1)
    search_idx <- which(grid_d1 >= trim_lo & grid_d1 <= trim_hi)
    if (length(search_idx) > 0) {
      gam_transition <- grid_d1[search_idx[which.max(abs(d1[search_idx]))]]
    }
  }

  data.frame(
    pathway_key = pathway_key,
    davies_p_nlr = davies_p,
    aic_linear_nlr = AIC(lm0),
    aic_segmented_nlr = aic_seg,
    delta_aic_nlr = AIC(lm0) - aic_seg,
    transition_age_segmented_nlr = psi_est,
    transition_age_gam_nlr = gam_transition,
    stringsAsFactors = FALSE
  )
}
cat("fit_transition_age_nlr() を定義しました。\n")

In [ ]:
%%R
# --- 16-N-2: 代表4パスウェイに適用し、未補正の結果(transition_age_df)と比較 ---
results_list_nlr <- vector("list", length(pathways_selected))
names(results_list_nlr) <- names(pathways_selected)

for (i in seq_along(pathways_selected)) {
  pw_key <- names(pathways_selected)[i]
  cat(sprintf("[NLR補正 %d/%d] %s\n", i, length(pathways_selected), pw_key))
  results_list_nlr[[pw_key]] <- tryCatch(
    fit_transition_age_nlr(pw_key, pathway_scores[pw_key, ], meta_sub$Age, meta_sub$Gender, meta_sub$NLR_proxy),
    error = function(e) { cat("  失敗:", conditionMessage(e), "\n"); NULL }
  )
}

transition_age_df_nlr <- do.call(rbind, Filter(Negate(is.null), results_list_nlr))
rownames(transition_age_df_nlr) <- NULL

compare_nlr <- merge(
  transition_age_df[, c("pathway_key", "category", "davies_p", "davies_padj_BH",
                          "delta_aic", "transition_age_segmented", "transition_age_gam")],
  transition_age_df_nlr,
  by = "pathway_key"
)
compare_nlr$davies_padj_BH_nlr <- p.adjust(compare_nlr$davies_p_nlr, method = "BH")

options(digits = 4)
cat("\n=== NLR_proxy補正前後の比較(代表4パスウェイ) ===\n")
print(compare_nlr[, c("category", "pathway_key", "davies_p", "davies_padj_BH", "delta_aic",
                       "davies_p_nlr", "davies_padj_BH_nlr", "delta_aic_nlr")])

write.csv(compare_nlr, file.path(out_dir, "OEP001041_NLRproxy_TransitionAge_sensitivity.csv"), row.names = FALSE)
cat("\n保存しました:", file.path(out_dir, "OEP001041_NLRproxy_TransitionAge_sensitivity.csv"), "\n")

cat("\n解釈の目安: davies_p / delta_aic が補正前後で同様の傾向(有意な離散breakpointを支持しない、\n",
    "あるいは支持する)を保つ場合、3.4節の『不連続transitionではなく緩やかなdriftと整合的』という\n",
    "結論はNLR_proxyで捉えられる血球組成シフトに単純に還元されないことを示唆する。\n", sep = "")